- баєс забраний з графіку + оцінка на сходках

In [ ]:
import os
import re
import csv
import time
import random
import shutil
import warnings
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.ensemble import IsolationForest
from sklearn.metrics import r2_score
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(False)
else:
    torch.set_num_threads(min(8, os.cpu_count() or 1))


DYN_METHOD_NAME = "Robust Dyn-Dur"
RM_METHOD_NAME = "Robbins-Monro Adaptive"
ACI_METHOD_NAME = "Adaptive Conformal (ACI)"
EWMA_ADP_METHOD_NAME = "EWMA Adaptive Threshold"
TDV_METHOD_NAME = "Adaptive Trend-Deriv-Var"
CORR_METHOD_NAME = "Diff Corridor Hybrid"
IF_METHOD_NAME = "Isolation Forest"

ALL_BASE_METHODS = [
    "Median / MAD",
    "Robust Z-Score",
    "AR",
    "Huber Regression",
    "Static Conformal Prediction",
    "Quantile Thresholding",
    "CUSUM",
    "NMS",
    DYN_METHOD_NAME,
    RM_METHOD_NAME,
    ACI_METHOD_NAME,
    EWMA_ADP_METHOD_NAME,
    TDV_METHOD_NAME,
    CORR_METHOD_NAME,
    IF_METHOD_NAME,
]
CONSENSUS_METHOD_NAME = "Local-Vol Corridor Consensus"
ALL_METHOD_NAMES = ALL_BASE_METHODS + [CONSENSUS_METHOD_NAME]

CORE_ROBUST_METHODS = [
    CONSENSUS_METHOD_NAME,
    "NMS",
    DYN_METHOD_NAME,
    "Median / MAD",
    IF_METHOD_NAME,
]


def _set_global_seeds(seed: int) -> None:
    """Set seeds across Python random, NumPy, and PyTorch (CPU and GPU)."""
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed(int(seed))
        torch.cuda.manual_seed_all(int(seed))


def _safe_r2_non_nan(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    min_var: float,
    y_fallback: Optional[np.ndarray] = None,
    pred_fallback: Optional[np.ndarray] = None,
) -> float:
    """Compute R2 safely without NaN: uses target subset when variance is sufficient, else fallback."""
    yt = np.asarray(y_true, dtype=np.float64)
    yp = np.asarray(y_pred, dtype=np.float64)
    if len(yt) >= 2 and float(np.var(yt)) >= float(min_var):
        val = float(r2_score(yt, yp))
        return float(np.clip(val, -10.0, 1.0)) if np.isfinite(val) else 0.0

    if y_fallback is not None and pred_fallback is not None:
        yf = np.asarray(y_fallback, dtype=np.float64)
        pf = np.asarray(pred_fallback, dtype=np.float64)
        if len(yf) >= 2 and float(np.var(yf)) >= float(min_var):
            val_f = float(r2_score(yf, pf))
            return float(np.clip(val_f, -10.0, 1.0)) if np.isfinite(val_f) else 0.0

    mse_flat = float(np.mean((yt - yp) ** 2)) if len(yt) > 0 else 0.0
    return 1.0 if mse_flat <= 1e-9 else 0.0


# ==============================================================================
# 1. CONFIGURATION (SINGLE TARGET INPUT + STEP TRANSITION EVALUATION)
# ==============================================================================
@dataclass
class Config:
    base_dir: Path = Path("/content/drive/MyDrive/DATA/My_Datasets_3client")
    results_rel_dir: str = "DATA/Results"
    fallback_results_dir: Path = Path("/tmp/AWMAD_Results")
    num_clients: int = 3
    fl_modes: Tuple[str, ...] = ("local_only", "fedavg", "fedprox_inv_loss")
    seeds: Tuple[int, ...] = (42, 1042, 2042)

    trim_head_points: int = 1000
    enable_resample: bool = False
    resample_rule: str = "10s"
    resample_points: Optional[int] = None

    stream_chunk_size: int = 5000
    window_size: int = 600
    step_size: int = 600
    decay_gamma: float = 0.85
    vote_threshold: float = 0.30

    lstm_seq_len: int = 30
    batch_train: int = 64
    batch_eval: int = 256
    max_epochs: int = 8
    warmup_epochs: int = 8
    oos_epochs: int = 6
    min_epochs: int = 3
    early_stop_rel_tol: float = 0.015
    early_stop_patience: int = 2
    fl_epoch_sync_spread: int = 1
    lr: float = 1e-3

    fl_max_update_norm: float = 1.50
    local_max_update_norm: float = 1.50
    local_anchor_mu: float = 0.03
    fedprox_mu: float = 0.03
    fedprox_loss_tau: float = 0.30
    fedprox_use_prox_term: bool = True
    fedprox_use_inv_loss_weight: bool = True

    robust_scale_std_ratio: float = 0.25
    robust_scale_min_floor: float = 1e-3

    hidden: int = 64
    dropout: float = 0.1
    num_layers: int = 2
    residual: bool = True
    seed: int = 42

    enable_math_method_pruning: bool = True
    qss_min_threshold: float = 0.14
    qss_min_f1_floor: float = 0.08
    qss_max_neg_fpr_ceiling: float = 0.25
    qss_max_stream_ratio: float = 0.16
    qss_stream_penalty_lambda: float = 8.0

    use_amp_cuda: bool = True
    show_per_method_plots: bool = False
    save_per_method_plots: bool = True
    save_per_method_plots_only_primary_seed: bool = True
    plot_persistence_on_mse_curves: bool = False

    min_r2_variance: float = 1e-6
    clean_eval_guard_radius: int = 2
    transition_mad_mult: float = 2.0
    transition_guard_radius: int = 2

    repair_method_source: str = CONSENSUS_METHOD_NAME
    max_repair_ratio: float = 0.20
    repair_short_gap_max: int = 3
    repair_anchor_radius: int = 2
    repair_corridor_mad_k: float = 3.5
    repair_edge_taper_len: int = 1
    repair_long_pred_weight: float = 0.25

    enabled_methods: Dict[str, bool] = field(
        default_factory=lambda: {
            "Median / MAD": True,
            "Robust Z-Score": False,
            "AR": True,
            "Huber Regression": False,
            "Static Conformal Prediction": False,
            "Quantile Thresholding": True,
            "CUSUM": False,
            "NMS": True,
            DYN_METHOD_NAME: True,
            RM_METHOD_NAME: False,
            ACI_METHOD_NAME: False,
            EWMA_ADP_METHOD_NAME: False,
            TDV_METHOD_NAME: True,
            CORR_METHOD_NAME: False,
            IF_METHOD_NAME: True,
            CONSENSUS_METHOD_NAME: True,
        }
    )

    mad_k: float = 1.65
    rzscore_thresh: float = 1.65
    ar_order: int = 5
    ar_thresh: float = 3.1
    huber_thresh: float = 2.9
    huber_max_iter: int = 200
    conformal_alpha: float = 0.04
    quantile_q: float = 0.95
    cusum_k: float = 0.45
    cusum_h: float = 3.5
    nms_radius: int = 10
    nms_quantile: float = 0.93
    nms_event_tolerance: int = 5
    if_contamination: float = 0.06
    if_estimators: int = 60

    dyn_adaptive_from_signal: bool = True
    dyn_boost_bounds: Tuple[float, float] = (1.6, 3.2)
    dyn_joint_bonus: float = 0.45
    dyn_pers_bonus: float = 0.55
    dyn_lstm_weight: float = 0.35
    dyn_smooth_win: int = 5
    dyn_var_win: int = 20
    dyn_slope_k: float = 0.8
    dyn_min_dur: int = 4
    dyn_pers_alpha: float = 0.75
    dyn_lvl_thresh: float = 1.5
    dyn_slope_thresh: float = 1.4
    dyn_var_thresh: float = 1.5
    dyn_peak_mad_k: float = 3.0
    dyn_peak_quantile_bounds: Tuple[float, float] = (0.93, 0.97)

    rm_init_thresh: float = 1.35
    rm_target_alpha: float = 0.06
    rm_step_lr: float = 0.10
    rm_thresh_bounds: Tuple[float, float] = (0.85, 2.50)

    aci_base_alpha: float = 0.06
    aci_gamma: float = 0.04
    aci_alpha_bounds: Tuple[float, float] = (0.015, 0.20)

    ewma_alpha: float = 0.12
    ewma_k_mult: float = 2.45
    ewma_cut_lift: float = 1.25
    ewma_level_boost: float = 5.50

    tdv_trend_win: int = 40
    tdv_slope_win: int = 20
    tdv_smooth_win: int = 5
    tdv_weights: Tuple[float, float] = (0.70, 0.30)
    tdv_tail_quantile: float = 0.96
    tdv_mad_mult: float = 2.8
    tdv_nms_radius: int = 2

    corr_smooth_win: int = 5
    corr_k_candidates: Tuple[float, ...] = (2.6, 3.0, 3.4, 3.8, 4.4)
    corr_default_k: float = 3.0
    corr_use_control_calibration: bool = True

    consensus_min_votes: int = 2
    consensus_min_vote_ratio: float = 0.25
    consensus_lstm_z_thresh: float = 2.3
    consensus_dual_norm_thresh: float = 0.90
    consensus_strong_corridor_norm: float = 1.30
    consensus_strong_lstm_norm: float = 1.40
    consensus_level_thresh: float = 1.15
    consensus_instab_slope: float = 0.25
    regime_gate_strength: float = 0.25
    corridor_center_radius: int = 4
    corridor_vol_win: int = 25
    corridor_k_base: float = 3.1
    corridor_tail_alpha: float = 0.30
    corridor_k_bounds: Tuple[float, float] = (2.5, 4.6)
    corridor_sigma_floor_ratio: float = 0.60

    run_control_validation: bool = True
    ctrl_window_indices: Tuple[int, ...] = (0, 6, 12, 18, 24)
    ctrl_n_spikes: int = 12
    ctrl_spike_sigma_mult: float = 6.5
    ctrl_include_zero_out_scenario: bool = True
    ctrl_base_clean_mad_k: float = 4.5

    target_col: str = "fh6000ens_energy_signalsbattery_value_64"
    voltage_col: str = "fh6000ens_energy_signalsbattery_cell_voltage_63"
    log_txt_name: str = "fl_anomalies_step_log.txt"
    log_csv_name: str = "fl_anomalies_step_log.csv"
    ctrl_csv_name: str = "control_anomalies_validation.csv"
    ctrl_summary_csv_name: str = "control_methods_summary.csv"
    wilcoxon_csv_name: str = "wilcoxon_stats_summary.csv"
    methods_eff_csv_name: str = "methods_effectiveness_matrix.csv"
    fl_summary_csv_name: str = "fl_modes_comparison_summary.csv"
    excluded_methods_csv_name: str = "excluded_noisy_methods_report.csv"
    q1_summary_txt_name: str = "q1_manuscript_ready_summary.txt"


def get_active_methods(cfg: Config) -> List[str]:
    return [m for m in ALL_METHOD_NAMES if cfg.enabled_methods.get(m, False)]


def get_active_base_methods(cfg: Config) -> List[str]:
    return [m for m in ALL_BASE_METHODS if cfg.enabled_methods.get(m, False)]


def get_explicitly_excluded_methods(cfg: Config) -> List[str]:
    return [m for m in ALL_METHOD_NAMES if not cfg.enabled_methods.get(m, False)]


def mode_exchanges_params(mode: str) -> bool:
    return mode in ("fedavg", "fedprox_inv_loss")


def _sanitize_1d(arr: np.ndarray) -> np.ndarray:
    a = np.asarray(arr, dtype=np.float64).ravel()
    if a.size == 0:
        return a
    if not np.all(np.isfinite(a)):
        a = pd.Series(a).interpolate(method="linear").ffill().bfill().to_numpy(dtype=np.float64)
        a = np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)
    return a


def _mad_scale(arr: np.ndarray, eps: float = 1e-6) -> Tuple[float, float]:
    med = float(np.median(arr))
    mad = 1.4826 * float(np.median(np.abs(arr - med)))
    return med, max(mad, eps)


def filter_methods_mathematically(
    eff_df: pd.DataFrame,
    cfg: Config,
) -> Tuple[List[str], pd.DataFrame]:
    """Evaluate detectors via QSS_m = CtrlScore_m * OccPenalty_m."""
    if eff_df.empty or not getattr(cfg, "enable_math_method_pruning", True):
        return get_active_methods(cfg), pd.DataFrame()

    agg = (
        eff_df.groupby("method", as_index=False)
        .agg(
            avg_f1=("mean_f1", "mean"),
            avg_event_f1=("mean_event_f1", lambda s: float(np.nanmean(s)) if "mean_event_f1" in eff_df.columns else 0.0),
            avg_prec=("mean_precision", "mean"),
            avg_neg_fpr=("mean_neg_fpr", "mean"),
            avg_stream_ratio=("stream_pct_of_series", lambda s: float(np.nanmean(s)) / 100.0),
        )
    )

    rows: List[Dict[str, object]] = []
    kept_methods: List[str] = []

    for _, r in agg.iterrows():
        m_name = str(r["method"])
        f1_pt = float(r["avg_f1"]) if np.isfinite(r["avg_f1"]) else 0.0
        f1_ev = float(r["avg_event_f1"]) if np.isfinite(r["avg_event_f1"]) else f1_pt
        f1_v = max(f1_pt, f1_ev) if m_name == "NMS" else f1_pt

        pr_v = float(r["avg_prec"]) if np.isfinite(r["avg_prec"]) else 0.0
        fpr_v = float(r["avg_neg_fpr"]) if np.isfinite(r["avg_neg_fpr"]) else 1.0
        occ_r = float(r["avg_stream_ratio"]) if np.isfinite(r["avg_stream_ratio"]) else 0.0

        ctrl_score = 0.55 * f1_v + 0.25 * pr_v + 0.20 * (1.0 - np.clip(fpr_v, 0.0, 1.0))
        excess_occ = max(0.0, occ_r - float(cfg.qss_max_stream_ratio))
        occ_penalty = float(np.exp(-float(cfg.qss_stream_penalty_lambda) * excess_occ))
        qss = float(ctrl_score * occ_penalty)

        reasons: List[str] = []
        if qss < float(cfg.qss_min_threshold):
            reasons.append(f"QSS={qss:.4f}<{cfg.qss_min_threshold}")
        if f1_v < float(cfg.qss_min_f1_floor):
            reasons.append(f"F1={f1_v:.4f}<{cfg.qss_min_f1_floor}")
        if fpr_v > float(cfg.qss_max_neg_fpr_ceiling) and f1_v < 0.55:
            reasons.append(f"NegFPR={fpr_v:.4f}>{cfg.qss_max_neg_fpr_ceiling}")
        if occ_r > float(cfg.qss_max_stream_ratio) and occ_penalty < 0.80:
            reasons.append(f"HighStreamOcc={occ_r:.2%}>{cfg.qss_max_stream_ratio:.0%}")

        keep = len(reasons) == 0
        if keep:
            kept_methods.append(m_name)

        rows.append(
            {
                "method": m_name,
                "exclusion_stage": "math_qss_pruning",
                "avg_f1": round(f1_pt, 4),
                "avg_event_f1": round(f1_ev, 4),
                "avg_precision": round(pr_v, 4),
                "avg_neg_fpr": round(fpr_v, 4),
                "stream_ratio": round(occ_r, 4),
                "ctrl_score": round(ctrl_score, 4),
                "occ_penalty": round(occ_penalty, 4),
                "qss": round(qss, 4),
                "kept_by_math": bool(keep),
                "rejection_reason": "; ".join(reasons) if not keep else "passed",
            }
        )

    qss_df = pd.DataFrame(rows).sort_values("qss", ascending=False)
    return kept_methods, qss_df


def build_excluded_methods_report(cfg: Config, qss_df: pd.DataFrame) -> pd.DataFrame:
    records: List[Dict[str, object]] = []
    for m_name in get_explicitly_excluded_methods(cfg):
        records.append(
            {
                "method": m_name,
                "exclusion_type": "config_disabled_in_enabled_methods",
                "qss": 0.0,
                "avg_f1": 0.0,
                "avg_neg_fpr": 0.0,
                "stream_ratio": 0.0,
                "reason": "enabled_methods[method] == False",
            }
        )

    if not qss_df.empty and "kept_by_math" in qss_df.columns:
        pruned = qss_df[qss_df["kept_by_math"] == False]
        for _, r in pruned.iterrows():
            records.append(
                {
                    "method": str(r["method"]),
                    "exclusion_type": "math_qss_pruned",
                    "qss": float(r["qss"]),
                    "avg_f1": float(r["avg_f1"]),
                    "avg_neg_fpr": float(r["avg_neg_fpr"]),
                    "stream_ratio": float(r["stream_ratio"]),
                    "reason": str(r["rejection_reason"]),
                }
            )

    return pd.DataFrame(records)


def compute_distinct_anomalies_mask(
    masks_by_method: Dict[str, np.ndarray],
    method_names: List[str],
    n_total: int,
) -> Tuple[np.ndarray, np.ndarray]:
    distinct_mask = np.zeros(n_total, dtype=bool)
    for m in method_names:
        if m in masks_by_method:
            m_arr = np.asarray(masks_by_method[m], dtype=bool)
            if len(m_arr) == n_total:
                distinct_mask |= m_arr
    return distinct_mask, np.flatnonzero(distinct_mask)


def _safe_savefig(
    fig: plt.Figure,
    target_path: Path,
    fallback_dir: Path = Path("/tmp/AWMAD_Results"),
    dpi: int = 150,
) -> Path:
    try:
        target_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(target_path, dpi=dpi)
        return target_path
    except OSError:
        fallback_dir.mkdir(parents=True, exist_ok=True)
        alt_path = fallback_dir / target_path.name
        fig.savefig(alt_path, dpi=dpi)
        return alt_path


def _sync_fallback_to_results(fallback_dir: Path, primary_results_dir: Path) -> None:
    if not fallback_dir.is_dir() or fallback_dir.resolve() == primary_results_dir.resolve():
        return
    try:
        primary_results_dir.mkdir(parents=True, exist_ok=True)
        for item in fallback_dir.iterdir():
            if item.is_file():
                shutil.copy2(item, primary_results_dir / item.name)
    except OSError:
        pass


# ==============================================================================
# 2. VARLIST SCHEMA & MULTI-CLIENT DATA LOADING
# ==============================================================================
def clean(s: str) -> str:
    s = re.sub(r"\[([^\]]+)\]\s*-?\s*", r"\1_", s)
    s = re.sub(r"[^\w\s]", "", s)
    return re.sub(r"[\s_]+", "_", s).strip("_").lower()


def _resolve_base_and_varlist(cfg: Config) -> Tuple[Path, Path]:
    try:
        from google.colab import drive  # type: ignore
        if not os.path.exists("/content/drive/MyDrive"):
            drive.mount("/content/drive", force_remount=False)
    except ImportError:
        pass

    candidate_bases = [
        cfg.base_dir,
        Path("/content/drive/MyDrive/DATA/My_Datasets_3client"),
        Path("DATA/My_Datasets_3client"),
        Path("data"),
    ]
    base = next((b for b in candidate_bases if b.is_dir()), cfg.base_dir)

    for vname in ("varlist.txt", "var_list.txt"):
        vp = base / vname
        if vp.is_file():
            return base, vp

    raise FileNotFoundError(f"Could not find varlist.txt or var_list.txt in '{base}'.")


def _resolve_results_dir(base_data_dir: Path, cfg: Config) -> Path:
    if "/content/drive/MyDrive" in str(base_data_dir.resolve()):
        res_dir = Path("/content/drive/MyDrive") / cfg.results_rel_dir
    else:
        res_dir = Path(cfg.results_rel_dir)
    try:
        res_dir.mkdir(parents=True, exist_ok=True)
        return res_dir
    except OSError:
        cfg.fallback_results_dir.mkdir(parents=True, exist_ok=True)
        return cfg.fallback_results_dir


def build_col_names(varlist_path: Path) -> List[str]:
    with open(varlist_path, encoding="utf-8") as f:
        names = [x.strip() for x in f if x.strip()]
    names = [
        f"{clean(x)}_{i}"
        for i, x in enumerate(names, 1)
        if "[TMP]" not in x and "Scanner Warning Zone Active" not in x
    ]
    cols = ["timestamp"] + names
    cols[34:34] = [f"unlisted_fh_id_6000_{i}" for i in range(35, 40)]
    cols[174:174] = [f"unlisted_navigation_command_{i}" for i in range(175, 186)]
    return cols


def _resample_to_fixed_points(df: pd.DataFrame, target_points: int) -> pd.DataFrame:
    n_cur = len(df)
    if target_points <= 0 or n_cur == 0:
        return df.iloc[:0]
    if n_cur == target_points:
        return df.reset_index(drop=True)

    old_pos = np.linspace(0.0, 1.0, num=n_cur)
    new_pos = np.linspace(0.0, 1.0, num=target_points)

    out_data = {}
    for col in df.columns:
        vals = df[col].to_numpy(dtype=np.float64)
        finite = np.isfinite(vals)
        if not finite.any():
            out_data[col] = np.zeros(target_points, dtype=np.float64)
        else:
            out_data[col] = np.interp(new_pos, old_pos[finite], vals[finite])

    return pd.DataFrame(out_data)


def load_client(client_num: int, cols: List[str], base: Path, cfg: Config) -> pd.DataFrame:
    path = base / f"out_client{client_num}.csv"
    with open(path, newline="", encoding="utf-8") as f:
        assert len(cols) == len(next(csv.reader(f))), (
            f"Column count mismatch in '{path}': expected {len(cols)}."
        )
    df = pd.read_csv(path, header=None, names=cols)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"]).set_index("timestamp").sort_index()
    num_df = df.select_dtypes(include="number")

    trim_n = max(0, int(cfg.trim_head_points))
    if trim_n > 0:
        if len(num_df) > trim_n:
            num_df = num_df.iloc[trim_n:]
        else:
            num_df = num_df.iloc[0:0]

    if cfg.resample_points is not None:
        return _resample_to_fixed_points(num_df, int(cfg.resample_points))

    if cfg.enable_resample:
        return num_df.resample(cfg.resample_rule, closed="left", label="right").mean()

    return num_df


def _pick_column(df: pd.DataFrame, desired: str, fallback_substr: str) -> str:
    if desired in df.columns:
        return desired
    matches = [c for c in df.columns if fallback_substr in c]
    if matches:
        return matches[0]
    num_cols = list(df.select_dtypes(include="number").columns)
    return num_cols[0]


def load_and_prepare_clients(
    cfg: Config,
) -> Tuple[Path, Path, Dict[int, pd.DataFrame], List[np.ndarray], List[np.ndarray], str, str, int]:
    base, varlist_path = _resolve_base_and_varlist(cfg)
    results_dir = _resolve_results_dir(base, cfg)
    cols = build_col_names(varlist_path)
    clients_raw = {c: load_client(c, cols, base, cfg) for c in range(1, cfg.num_clients + 1)}

    voltage_col = _pick_column(clients_raw[1], cfg.voltage_col, "voltage")
    target_col = _pick_column(clients_raw[1], cfg.target_col, "battery_value")

    if cfg.resample_points is not None:
        mode_desc = f"points={cfg.resample_points}, head_trimmed={cfg.trim_head_points}"
    elif cfg.enable_resample:
        mode_desc = f"resampled={cfg.resample_rule}, head_trimmed={cfg.trim_head_points}"
    else:
        mode_desc = f"no_resample, head_trimmed={cfg.trim_head_points}"

    fig, axes = plt.subplots(cfg.num_clients, 1, figsize=(14, 3.0 * cfg.num_clients), sharex=False)
    if cfg.num_clients == 1:
        axes = [axes]
    for ax, c in zip(axes, range(1, cfg.num_clients + 1)):
        v_c = _pick_column(clients_raw[c], voltage_col, "voltage")
        s_v = clients_raw[c][v_c]
        ax.plot(clients_raw[c].index, s_v.values, linewidth=1.1, label=f"Client {c} ({v_c})")
        ax.set_title(f"Client {c} — Cell Voltage ({mode_desc}, n={len(s_v):,})")
        ax.set_ylabel("Voltage")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="upper right")
    axes[-1].set_xlabel("Index / Timestamp")
    fig.suptitle(f"Battery Cell Voltage Across Clients ({mode_desc}): {voltage_col}", fontsize=13, y=1.01)
    plt.tight_layout()
    _safe_savefig(fig, results_dir / "clients_voltage_overview.png", cfg.fallback_results_dir, dpi=150)
    plt.close(fig)

    min_len = min(len(df) for df in clients_raw.values())
    if min_len < cfg.window_size:
        usable_len = 0
    else:
        n_steps = max(0, (min_len - cfg.window_size) // cfg.step_size) + 1
        usable_len = min(min_len, cfg.window_size + (n_steps - 1) * cfg.step_size)

    clients_trunc = {
        c: df.iloc[:usable_len].ffill().bfill().fillna(0.0)
        for c, df in clients_raw.items()
    }
    signals_1d = [_sanitize_1d(clients_trunc[c][target_col].to_numpy(dtype=np.float64)) for c in range(1, cfg.num_clients + 1)]
    voltages_1d = [
        _sanitize_1d(clients_trunc[c][_pick_column(clients_trunc[c], voltage_col, "voltage")].to_numpy(dtype=np.float64))
        for c in range(1, cfg.num_clients + 1)
    ]
    return base, results_dir, clients_trunc, signals_1d, voltages_1d, voltage_col, target_col, usable_len


# ==============================================================================
# 3. ROBUST SCALE PROTECTION & UNIVARIATE LSTM FORECAST NETWORKS
# ==============================================================================
def _robust_fit_transform(
    series_1d: np.ndarray,
    std_ratio_floor: float = 0.25,
    min_floor: float = 1e-3,
) -> Tuple[np.ndarray, float, float]:
    arr = _sanitize_1d(series_1d)
    if len(arr) == 0:
        return np.zeros(0, dtype=np.float64), 0.0, 1.0
    med = float(np.median(arr))
    mad = 1.4826 * float(np.median(np.abs(arr - med)))
    std_fb = float(np.std(arr))
    iqr = float(np.percentile(arr, 75.0) - np.percentile(arr, 25.0)) / 1.349

    best_sc = max(mad, iqr, float(std_ratio_floor) * std_fb)
    if best_sc > float(min_floor):
        scale = best_sc
    elif std_fb > float(min_floor):
        scale = std_fb
    else:
        scale = 1.0
    return (arr - med) / scale, med, scale


def _build_seq_tensors(scaled_1d: np.ndarray, seq_len: int) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]:
    n_samples = max(0, len(scaled_1d) - seq_len)
    if n_samples == 0:
        return None, None
    s_t = torch.tensor(scaled_1d, dtype=torch.float32)
    idx_base = torch.arange(n_samples)[:, None] + torch.arange(seq_len)[None, :]
    x_win = s_t[idx_base].unsqueeze(-1)
    y_win = s_t[seq_len:].unsqueeze(-1)
    return x_win, y_win


class Forecast(nn.Module):
    def __init__(
        self,
        inputs: int = 1,
        hidden: int = 64,
        dropout: float = 0.1,
        residual: bool = True,
        num_layers: int = 2,
    ):
        super().__init__()
        self.residual = residual
        self.lstm = nn.LSTM(
            inputs,
            hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, 1))
        if residual:
            nn.init.zeros_(self.head[-1].weight)
            nn.init.zeros_(self.head[-1].bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z, _ = self.lstm(x)
        p = self.head(z[:, -1])
        return p + x[:, -1, -1:] if self.residual else p


def _build_forecast_model(cfg: Config) -> Forecast:
    return Forecast(
        inputs=1,
        hidden=cfg.hidden,
        dropout=cfg.dropout,
        residual=cfg.residual,
        num_layers=cfg.num_layers,
    ).to(DEVICE)


def _clone_state_dict_cpu(sd: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in sd.items()}


def _clamp_local_update(
    base_sd: Dict[str, torch.Tensor],
    updated_sd: Dict[str, torch.Tensor],
    max_norm: float,
) -> Dict[str, torch.Tensor]:
    if max_norm <= 0.0:
        return updated_sd
    sq_sum = 0.0
    for k in base_sd:
        diff = updated_sd[k].float() - base_sd[k].float()
        sq_sum += float(torch.sum(diff * diff).item())
    total_norm = float(np.sqrt(sq_sum))
    if total_norm <= max_norm or total_norm < 1e-12:
        return updated_sd
    scale = max_norm / total_norm
    return {
        k: (base_sd[k].float() + scale * (updated_sd[k].float() - base_sd[k].float())).to(base_sd[k].dtype)
        for k in base_sd
    }


def _state_dict_weighted_avg(weighted_items: List[Tuple[Dict[str, torch.Tensor], float]]) -> Dict[str, torch.Tensor]:
    total_w = sum(w for _, w in weighted_items) + 1e-12
    avg = {k: torch.zeros_like(v, dtype=torch.float32) for k, v in weighted_items[0][0].items()}
    for sd, w in weighted_items:
        scale = float(w) / total_w
        for k in avg:
            avg[k].add_(sd[k].float(), alpha=scale)
    return avg


def _amp_context(cfg: Config):
    if DEVICE == "cuda" and cfg.use_amp_cuda:
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        return torch.amp.autocast("cuda", dtype=dtype)
    return torch.amp.autocast("cpu", enabled=False)


def _prepare_cached_windows(
    client_windows: List[np.ndarray],
    cfg: Config,
) -> List[Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]]:
    cached: List[Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]] = []
    for win in client_windows:
        win_scaled, _, _ = _robust_fit_transform(
            win,
            std_ratio_floor=cfg.robust_scale_std_ratio,
            min_floor=cfg.robust_scale_min_floor,
        )
        cached.append(_build_seq_tensors(win_scaled, seq_len=cfg.lstm_seq_len))
    return cached


def canonical_server_fedavg_step(
    server_global_sd: Dict[str, torch.Tensor],
    client_windows: List[np.ndarray],
    cfg: Config,
    epochs_to_run: int,
    seed_offset: int = 9000,
    reusable_model: Optional[Forecast] = None,
    prepared_tensors: Optional[List[Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]]] = None,
) -> Tuple[Dict[str, torch.Tensor], List[float], List[int]]:
    model = reusable_model if reusable_model is not None else _build_forecast_model(cfg)
    batch_sz = max(1, int(cfg.batch_train))
    n_epochs = max(1, int(epochs_to_run))
    min_ep = max(1, int(cfg.min_epochs))
    rel_tol = max(0.0, float(cfg.early_stop_rel_tol))
    patience = max(1, int(cfg.early_stop_patience))

    seq_cache = (
        prepared_tensors
        if prepared_tensors is not None
        else _prepare_cached_windows(client_windows, cfg)
    )

    client_updates: List[Tuple[Dict[str, torch.Tensor], float]] = []
    client_losses: List[float] = []
    epochs_done: List[int] = []

    for c_idx, (x_win, y_win) in enumerate(seq_cache):
        if x_win is None or y_win is None or len(x_win) == 0:
            client_updates.append((_clone_state_dict_cpu(server_global_sd), 1.0))
            client_losses.append(0.0)
            epochs_done.append(0)
            continue

        torch.manual_seed(cfg.seed + seed_offset + c_idx)
        model.load_state_dict({k: v.to(DEVICE).float() for k, v in server_global_sd.items()})

        x_gpu = x_win.to(DEVICE)
        y_gpu = y_win.to(DEVICE)
        n_samples = len(x_gpu)

        opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-4)
        model.train()
        last_loss = 1.0
        prev_ep_loss = float("inf")
        stagnant_ep = 0
        ran_ep = 0

        for ep in range(n_epochs):
            ran_ep = ep + 1
            perm = torch.randperm(n_samples, device=DEVICE)
            batch_losses = []
            for b_start in range(0, n_samples, batch_sz):
                b_idx = perm[b_start : b_start + batch_sz]
                xb, yb = x_gpu[b_idx], y_gpu[b_idx]
                opt.zero_grad(set_to_none=True)
                with _amp_context(cfg):
                    pred = model(xb)
                    loss = nn.functional.mse_loss(pred.float(), yb.float())
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                batch_losses.append(float(loss.item()))
            if batch_losses:
                ep_mse = float(np.mean(batch_losses))
                if not np.isfinite(ep_mse):
                    ep_mse = prev_ep_loss if np.isfinite(prev_ep_loss) else 1.0
                last_loss = ep_mse + 1e-6
                if (ep + 1) >= min_ep and np.isfinite(prev_ep_loss):
                    rel_gain = (prev_ep_loss - last_loss) / max(abs(prev_ep_loss), 1e-8)
                    if rel_gain < rel_tol:
                        stagnant_ep += 1
                        if stagnant_ep >= patience:
                            break
                    else:
                        stagnant_ep = 0
                prev_ep_loss = last_loss

        sd_cpu = _clamp_local_update(server_global_sd, _clone_state_dict_cpu(model.state_dict()), cfg.fl_max_update_norm)
        client_updates.append((sd_cpu, float(n_samples)))
        client_losses.append(last_loss)
        epochs_done.append(ran_ep)

    return _state_dict_weighted_avg(client_updates), client_losses, epochs_done


def federated_lstm_step(
    global_sd: Dict[str, torch.Tensor],
    client_windows: List[np.ndarray],
    mode: str,
    cfg: Config,
    epochs_to_run: int,
    seed_offset: int = 0,
    reusable_model: Optional[Forecast] = None,
    local_models_sd: Optional[List[Dict[str, torch.Tensor]]] = None,
    prepared_tensors: Optional[List[Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]]] = None,
) -> Tuple[Dict[str, torch.Tensor], List[float], List[int], bool]:
    if mode not in ("local_only", "fedavg", "fedprox_inv_loss"):
        raise ValueError(f"Unsupported FL mode: {mode}")

    exchange_flag = mode_exchanges_params(mode)
    global_tensors = {k: v.to(DEVICE).float() for k, v in global_sd.items()}
    local_states: List[Dict[str, torch.Tensor]] = []
    sample_counts: List[float] = []
    client_losses: List[float] = []
    epochs_done: List[int] = []
    model = reusable_model if reusable_model is not None else _build_forecast_model(cfg)
    batch_sz = max(1, int(cfg.batch_train))
    n_epochs = max(1, int(epochs_to_run))
    min_ep = max(1, int(cfg.min_epochs))
    rel_tol = max(0.0, float(cfg.early_stop_rel_tol))
    patience = max(1, int(cfg.early_stop_patience))
    anchor_mu = max(0.0, float(cfg.local_anchor_mu))

    seq_cache = (
        prepared_tensors
        if prepared_tensors is not None
        else _prepare_cached_windows(client_windows, cfg)
    )

    for c_idx, (x_win, y_win) in enumerate(seq_cache):
        if x_win is None or y_win is None or len(x_win) == 0:
            fallback_sd = (
                _clone_state_dict_cpu(local_models_sd[c_idx])
                if (not exchange_flag and local_models_sd is not None)
                else _clone_state_dict_cpu(global_sd)
            )
            local_states.append(fallback_sd)
            sample_counts.append(1.0)
            client_losses.append(0.0)
            epochs_done.append(0)
            continue

        torch.manual_seed(cfg.seed + seed_offset + c_idx)
        prev_sd_cpu = (
            _clone_state_dict_cpu(local_models_sd[c_idx])
            if (not exchange_flag and local_models_sd is not None)
            else _clone_state_dict_cpu(global_sd)
        )
        model.load_state_dict({k: v.to(DEVICE).float() for k, v in prev_sd_cpu.items()})
        anchor_tensors = (
            {k: v.to(DEVICE).float() for k, v in prev_sd_cpu.items()}
            if (not exchange_flag and anchor_mu > 0.0)
            else None
        )

        x_gpu = x_win.to(DEVICE)
        y_gpu = y_win.to(DEVICE)
        n_samples = len(x_gpu)

        opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-4)
        model.train()
        last_loss = 1.0
        prev_ep_loss = float("inf")
        stagnant_ep = 0
        ran_ep = 0

        for ep in range(n_epochs):
            ran_ep = ep + 1
            perm = torch.randperm(n_samples, device=DEVICE)
            batch_losses = []
            for b_start in range(0, n_samples, batch_sz):
                b_idx = perm[b_start : b_start + batch_sz]
                xb, yb = x_gpu[b_idx], y_gpu[b_idx]
                opt.zero_grad(set_to_none=True)
                with _amp_context(cfg):
                    pred = model(xb)
                    mse = nn.functional.mse_loss(pred.float(), yb.float())
                    if mode == "fedprox_inv_loss" and getattr(cfg, "fedprox_use_prox_term", True):
                        prox = sum(torch.sum((p.float() - global_tensors[n]) ** 2) for n, p in model.named_parameters())
                        loss = mse + 0.5 * cfg.fedprox_mu * prox
                    elif not exchange_flag and anchor_tensors is not None:
                        anc = sum(torch.sum((p.float() - anchor_tensors[n]) ** 2) for n, p in model.named_parameters())
                        loss = mse + 0.5 * anchor_mu * anc
                    else:
                        loss = mse
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                batch_losses.append(float(mse.item()))
            if batch_losses:
                ep_mse = float(np.mean(batch_losses))
                if not np.isfinite(ep_mse):
                    ep_mse = prev_ep_loss if np.isfinite(prev_ep_loss) else 1.0
                last_loss = ep_mse + 1e-6
                if (ep + 1) >= min_ep and np.isfinite(prev_ep_loss):
                    rel_gain = (prev_ep_loss - last_loss) / max(abs(prev_ep_loss), 1e-8)
                    if rel_gain < rel_tol:
                        stagnant_ep += 1
                        if stagnant_ep >= patience:
                            break
                    else:
                        stagnant_ep = 0
                prev_ep_loss = last_loss

        sd_cpu = _clone_state_dict_cpu(model.state_dict())
        if not exchange_flag:
            sd_cpu = _clamp_local_update(prev_sd_cpu, sd_cpu, cfg.local_max_update_norm)
            if local_models_sd is not None:
                local_models_sd[c_idx] = sd_cpu
        else:
            sd_cpu = _clamp_local_update(prev_sd_cpu, sd_cpu, cfg.fl_max_update_norm)

        local_states.append(sd_cpu)
        sample_counts.append(float(n_samples))
        client_losses.append(last_loss)
        epochs_done.append(ran_ep)

    if not exchange_flag:
        return _clone_state_dict_cpu(global_sd), client_losses, epochs_done, False

    pos_eps = [e for e in epochs_done if e > 0]
    if pos_eps and cfg.fl_epoch_sync_spread >= 0:
        med_ep = int(round(float(np.median(pos_eps))))
        spread = int(cfg.fl_epoch_sync_spread)
        lo_ep = max(min_ep, med_ep - spread)
        hi_ep = min(n_epochs, med_ep + spread)
        epochs_done = [int(np.clip(e, lo_ep, hi_ep)) if e > 0 else 0 for e in epochs_done]

    finite_losses = [x for x in client_losses if np.isfinite(x) and x > 0.0]
    loss_floor = float(np.median(finite_losses)) if finite_losses else 1e-3
    tau = float(np.clip(cfg.fedprox_loss_tau, 0.05, 1.0))

    collected: List[Tuple[Dict[str, torch.Tensor], float]] = []
    for sd_c, n_c, l_c in zip(local_states, sample_counts, client_losses):
        if mode == "fedprox_inv_loss" and getattr(cfg, "fedprox_use_inv_loss_weight", True) and np.isfinite(l_c):
            eff_loss = max(l_c, loss_floor, 1e-6)
            weight = n_c / (eff_loss ** tau)
        else:
            weight = n_c
        collected.append((sd_c, weight))

    return _state_dict_weighted_avg(collected), client_losses, epochs_done, True


def _build_model_agnostic_clean_mask(
    win: np.ndarray,
    seq_len: int,
    det_map: Dict[str, Tuple[np.ndarray, np.ndarray, float]],
    guard_radius: int,
) -> np.ndarray:
    x = _sanitize_1d(win)
    w_len = len(x)
    valid = np.zeros(w_len, dtype=bool)
    if w_len <= seq_len:
        return valid
    valid[seq_len:] = True

    union_anom = np.zeros(w_len, dtype=bool)
    for m_name in ("AR", "Median / MAD", CONSENSUS_METHOD_NAME):
        if m_name in det_map:
            _, sc_arr, _ = det_map[m_name]
            union_anom |= np.asarray(sc_arr, dtype=np.float64) > 0.0

    if np.any(union_anom):
        r = max(0, int(guard_radius))
        for idx in np.where(union_anom)[0]:
            lo = max(seq_len, idx - r)
            hi = min(w_len, idx + r + 1)
            valid[lo:hi] = False

    if int(np.sum(valid)) < 20:
        valid[seq_len:] = True
    return valid


def _build_step_transition_mask(
    win: np.ndarray,
    seq_len: int,
    mad_mult: float = 2.0,
    guard_radius: int = 2,
) -> np.ndarray:
    """Identify step-transition / level-change zones in the target series for focused regime-shift evaluation."""
    x = _sanitize_1d(win)
    n = len(x)
    t_mask = np.zeros(n, dtype=bool)
    if n <= max(2, seq_len):
        return t_mask

    diff1 = np.zeros(n, dtype=np.float64)
    diff1[1:] = np.abs(np.diff(x))
    nz_diffs = diff1[diff1 > 0.0]
    if len(nz_diffs) > 0:
        med_nz = float(np.median(nz_diffs))
        mad_nz = 1.4826 * float(np.median(np.abs(nz_diffs - med_nz)))
        thr = max(0.5 * med_nz, med_nz + 0.25 * float(mad_mult) * mad_nz, 1e-5)
    else:
        thr = 1e-5

    jump_idx = np.where((diff1 >= thr) & (np.arange(n) >= seq_len))[0]
    r = max(0, int(guard_radius))
    for idx in jump_idx:
        lo = max(seq_len, idx - r)
        hi = min(n, idx + r + 1)
        t_mask[lo:hi] = True

    return t_mask


def _compute_persistence_metrics(
    target_series: np.ndarray,
    seq_len: int,
    clean_eval_mask: np.ndarray,
    min_r2_var: float,
) -> Dict[str, float]:
    y = _sanitize_1d(target_series)
    n = len(y)
    if n <= max(1, seq_len):
        return {
            "pers_clean_mse": 0.0,
            "pers_clean_mae": 0.0,
            "pers_clean_r2": 1.0,
            "pers_all_valid_mse": 0.0,
            "pers_all_valid_mae": 0.0,
        }

    all_idx = np.arange(max(1, seq_len), n)
    true_all = y[all_idx]
    pred_all = y[all_idx - 1]
    diff_all = true_all - pred_all
    pers_all_mse = float(np.mean(diff_all * diff_all))
    pers_all_mae = float(np.mean(np.abs(diff_all)))

    eval_idx = np.where(clean_eval_mask & (np.arange(n) >= max(1, seq_len)))[0]
    if len(eval_idx) == 0:
        eval_idx = all_idx

    true_ev = y[eval_idx]
    pred_ev = y[eval_idx - 1]
    diff_ev = true_ev - pred_ev
    pers_mse = float(np.mean(diff_ev * diff_ev))
    pers_mae = float(np.mean(np.abs(diff_ev)))
    pers_r2 = _safe_r2_non_nan(true_ev, pred_ev, min_r2_var, true_all, pred_all)

    return {
        "pers_clean_mse": pers_mse,
        "pers_clean_mae": pers_mae,
        "pers_clean_r2": pers_r2,
        "pers_all_valid_mse": pers_all_mse,
        "pers_all_valid_mae": pers_all_mae,
    }


@torch.no_grad()
def lstm_window_predict_and_metrics(
    win_input: np.ndarray,
    model_sd: Dict[str, torch.Tensor],
    cfg: Config,
    eval_target: Optional[np.ndarray] = None,
    clean_eval_mask: Optional[np.ndarray] = None,
    transition_mask: Optional[np.ndarray] = None,
    reusable_model: Optional[Forecast] = None,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    win_clean_in = _sanitize_1d(win_input)
    target_series = win_clean_in if eval_target is None else _sanitize_1d(eval_target)
    w_len = len(win_clean_in)
    errs = np.zeros(w_len, dtype=np.float64)
    full_pred = win_clean_in.astype(np.float64).copy()

    n_pred = max(0, w_len - cfg.lstm_seq_len)
    if n_pred == 0 or len(target_series) < w_len:
        return errs, full_pred, {
            "n_forecast_points": 0,
            "mse": 0.0,
            "rmse": 0.0,
            "mae": 0.0,
            "r2": 1.0,
            "all_valid_mse": 0.0,
            "all_valid_mae": 0.0,
            "all_valid_r2": 1.0,
            "transition_pts": 0,
            "transition_mse": float("nan"),
            "transition_r2": float("nan"),
            "scale_used": 1.0,
            "target_var": 0.0,
        }

    win_scaled, med_s, scale_s = _robust_fit_transform(
        win_clean_in,
        std_ratio_floor=cfg.robust_scale_std_ratio,
        min_floor=cfg.robust_scale_min_floor,
    )
    x_win, _ = _build_seq_tensors(win_scaled, seq_len=cfg.lstm_seq_len)
    if x_win is None or len(x_win) == 0:
        return errs, full_pred, {
            "n_forecast_points": 0,
            "mse": 0.0,
            "rmse": 0.0,
            "mae": 0.0,
            "r2": 1.0,
            "all_valid_mse": 0.0,
            "all_valid_mae": 0.0,
            "all_valid_r2": 1.0,
            "transition_pts": 0,
            "transition_mse": float("nan"),
            "transition_r2": float("nan"),
            "scale_used": scale_s,
            "target_var": 0.0,
        }

    model = reusable_model if reusable_model is not None else _build_forecast_model(cfg)
    model.load_state_dict(model_sd)
    model.eval()

    batch_sz = max(1, int(cfg.batch_eval))
    preds_list = []
    x_gpu = x_win.to(DEVICE)
    for b_start in range(0, len(x_gpu), batch_sz):
        xb = x_gpu[b_start : b_start + batch_sz]
        with _amp_context(cfg):
            p_b = model(xb).float().cpu().numpy().ravel()
        preds_list.append(p_b)

    pred_scaled = _sanitize_1d(np.concatenate(preds_list))
    pred_orig = pred_scaled * scale_s + med_s
    full_pred[cfg.lstm_seq_len :] = pred_orig
    errs[cfg.lstm_seq_len :] = np.abs(target_series[cfg.lstm_seq_len :] - pred_orig)

    all_idx = np.arange(cfg.lstm_seq_len, w_len)
    true_all = target_series[all_idx]
    pred_all = full_pred[all_idx]
    diff_all = true_all - pred_all
    all_valid_mse = float(np.mean(diff_all * diff_all))
    all_valid_mae = float(np.mean(np.abs(diff_all)))
    all_valid_r2 = _safe_r2_non_nan(true_all, pred_all, float(cfg.min_r2_variance))

    if clean_eval_mask is not None and len(clean_eval_mask) == w_len:
        eval_idx = np.where(clean_eval_mask & (np.arange(w_len) >= cfg.lstm_seq_len))[0]
    else:
        eval_idx = all_idx

    if len(eval_idx) == 0:
        eval_idx = all_idx

    true_eval = target_series[eval_idx]
    pred_eval = full_pred[eval_idx]
    diff = true_eval - pred_eval

    mse = float(np.mean(diff * diff))
    rmse = float(np.sqrt(mse))
    mae = float(np.mean(np.abs(diff)))
    target_var = float(np.var(true_eval))
    r2 = _safe_r2_non_nan(true_eval, pred_eval, float(cfg.min_r2_variance), true_all, pred_all)

    # Focused evaluation on step-transition / regime-change segments
    if transition_mask is not None and len(transition_mask) == w_len:
        tr_idx = np.where(transition_mask & (np.arange(w_len) >= cfg.lstm_seq_len))[0]
    else:
        tr_idx = np.empty(0, dtype=int)

    if len(tr_idx) > 0:
        true_tr = target_series[tr_idx]
        pred_tr = full_pred[tr_idx]
        diff_tr = true_tr - pred_tr
        tr_mse = float(np.mean(diff_tr * diff_tr))
        tr_r2 = _safe_r2_non_nan(true_tr, pred_tr, float(cfg.min_r2_variance), true_all, pred_all)
    else:
        tr_mse = float("nan")
        tr_r2 = float("nan")

    return errs, full_pred, {
        "n_forecast_points": int(len(true_eval)),
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "all_valid_mse": all_valid_mse,
        "all_valid_mae": all_valid_mae,
        "all_valid_r2": all_valid_r2,
        "transition_pts": int(len(tr_idx)),
        "transition_mse": tr_mse,
        "transition_r2": tr_r2,
        "scale_used": scale_s,
        "target_var": target_var,
    }


# ==============================================================================
# 4. HYBRID WINDOW-LEVEL ANOMALY REPAIR & BOUNDARY SMOOTHING
# ==============================================================================
def _clean_anchor_value(
    series: np.ndarray,
    clean_mask: np.ndarray,
    center_idx: int,
    direction: int,
    radius: int,
) -> Optional[float]:
    n = len(series)
    vals: List[float] = []
    step = 1 if direction >= 0 else -1
    cur = center_idx + step
    while 0 <= cur < n and len(vals) < max(1, radius):
        if clean_mask[cur] and np.isfinite(series[cur]):
            vals.append(float(series[cur]))
        cur += step
    if not vals:
        return None
    return float(np.median(vals))


def _cap_repair_mask(
    anom_mask: np.ndarray,
    severity_scores: Optional[np.ndarray],
    max_ratio: float,
) -> np.ndarray:
    mask = np.asarray(anom_mask, dtype=bool).copy()
    n = len(mask)
    if n == 0 or not np.any(mask):
        return mask

    max_allowed = max(1, int(np.floor(np.clip(max_ratio, 0.01, 1.0) * n)))
    cur_idx = np.where(mask)[0]
    if len(cur_idx) <= max_allowed:
        return mask

    if severity_scores is not None and len(severity_scores) == n:
        order = cur_idx[np.argsort(-np.asarray(severity_scores, dtype=np.float64)[cur_idx])]
    else:
        order = cur_idx

    capped = np.zeros(n, dtype=bool)
    capped[order[:max_allowed]] = True
    return capped


def _apply_boundary_smoothing(
    original_series: np.ndarray,
    repaired_series: np.ndarray,
    anom_mask: np.ndarray,
    taper_len: int,
) -> np.ndarray:
    out = _sanitize_1d(repaired_series).copy()
    n = len(out)
    t_len = max(0, int(taper_len))
    if n == 0 or not np.any(anom_mask) or t_len == 0:
        return out

    i = 0
    while i < n:
        if not anom_mask[i]:
            i += 1
            continue
        start = i
        while i < n and anom_mask[i]:
            i += 1
        end = i
        seg_len = end - start

        if seg_len >= 2:
            v_l = out[start - 1] if start > 0 else out[start]
            v_r = out[end] if end < n else out[end - 1]
            bridge = np.linspace(v_l, v_r, num=seg_len + 2)[1:-1]
            k_edge = min(t_len, seg_len // 2)
            if k_edge > 0:
                w_l = 0.5 * (1.0 - np.cos(np.pi * np.linspace(0.0, 1.0, num=k_edge + 1)[1:]))
                out[start : start + k_edge] = (
                    w_l * bridge[:k_edge] + (1.0 - w_l) * out[start : start + k_edge]
                )
                w_r = 0.5 * (1.0 - np.cos(np.pi * np.linspace(1.0, 0.0, num=k_edge + 1)[:-1]))
                out[end - k_edge : end] = (
                    w_r * bridge[-k_edge:] + (1.0 - w_r) * out[end - k_edge : end]
                )

    return _sanitize_1d(out)


def repair_window_hybrid(
    win: np.ndarray,
    anom_mask: np.ndarray,
    cfg: Config,
    lstm_pred_series: Optional[np.ndarray] = None,
    severity_scores: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, int]:
    x = _sanitize_1d(win).copy()
    n = len(x)
    mask = _cap_repair_mask(anom_mask, severity_scores, cfg.max_repair_ratio)
    if n == 0 or not np.any(mask):
        return x, 0

    clean_mask = (~mask) & np.isfinite(x)
    if not np.any(clean_mask):
        med_all = float(np.nanmedian(x)) if np.any(np.isfinite(x)) else 0.0
        return np.full(n, med_all, dtype=np.float64), int(np.sum(mask))

    clean_vals = x[clean_mask]
    global_med = float(np.median(clean_vals))
    mad_scale = max(1.4826 * float(np.median(np.abs(clean_vals - global_med))), 1e-4)

    out = x.copy()
    i = 0
    while i < n:
        if not mask[i]:
            i += 1
            continue
        start = i
        while i < n and mask[i]:
            i += 1
        end = i
        seg_len = end - start

        left_val = _clean_anchor_value(out, clean_mask, start - 1, direction=-1, radius=cfg.repair_anchor_radius)
        right_val = _clean_anchor_value(out, clean_mask, end, direction=+1, radius=cfg.repair_anchor_radius)

        if left_val is not None and right_val is not None:
            anchor_ref = 0.5 * (left_val + right_val)
        elif left_val is not None:
            anchor_ref = left_val
        elif right_val is not None:
            anchor_ref = right_val
        else:
            anchor_ref = global_med

        v_l = left_val if left_val is not None else anchor_ref
        v_r = right_val if right_val is not None else anchor_ref

        if seg_len <= max(1, int(cfg.repair_short_gap_max)):
            if seg_len == 1:
                out[start] = 0.5 * (v_l + v_r)
            else:
                out[start:end] = np.linspace(v_l, v_r, num=seg_len + 2)[1:-1]
        else:
            t_rel = np.linspace(0.0, 1.0, num=seg_len)
            bridge = (1.0 - t_rel) * v_l + t_rel * v_r
            if lstm_pred_series is not None and len(lstm_pred_series) == n:
                seg_pred = _sanitize_1d(lstm_pred_series)[start:end].copy()
                w_p = float(np.clip(cfg.repair_long_pred_weight, 0.0, 0.6))
                blended = w_p * seg_pred + (1.0 - w_p) * bridge
                max_dev = max(cfg.repair_corridor_mad_k * mad_scale, 1e-4)
                out[start:end] = np.clip(blended, bridge - max_dev, bridge + max_dev)
            else:
                out[start:end] = bridge

    out = _apply_boundary_smoothing(x, out, mask, cfg.repair_edge_taper_len)
    return _sanitize_1d(out), int(np.sum(mask))


# ==============================================================================
# 5. ANOMALY DETECTORS & VOLTAGE-ONLY REGIME GATE
# ==============================================================================
def _make_lag_matrix(series: np.ndarray, lag: int) -> Tuple[np.ndarray, np.ndarray]:
    n = len(series)
    if n <= lag:
        return np.empty((0, lag), dtype=np.float64), np.empty((0,), dtype=np.float64)
    idx = np.arange(n - lag)[:, None] + np.arange(lag)[None, :]
    return series[idx], series[lag:]


def _nms_1d(scores: np.ndarray, radius: int = 10, thresh_quantile: float = 0.93) -> np.ndarray:
    n = len(scores)
    out = np.zeros(n, dtype=np.float64)
    if n == 0:
        return out
    thresh = float(np.quantile(scores, thresh_quantile))
    cands = np.flatnonzero(scores >= thresh)
    if len(cands) == 0:
        return out
    order = cands[np.argsort(-scores[cands])]
    suppressed = np.zeros(n, dtype=bool)
    for idx in order:
        if not suppressed[idx]:
            out[idx] = scores[idx]
            suppressed[max(0, idx - radius) : min(n, idx + radius + 1)] = True
    return out


def _robust_z_from_errs(errs: np.ndarray, local_win: int = 25, eps: float = 1e-6) -> np.ndarray:
    e = _sanitize_1d(errs)
    n = len(e)
    if n == 0:
        return np.zeros(0, dtype=np.float64)

    glob_med, glob_mad = _mad_scale(e, eps=eps)
    w_loc = max(5, min(int(local_win), n))
    s_err = pd.Series(e)
    loc_med = s_err.rolling(window=w_loc, min_periods=1, center=True).median().to_numpy(dtype=np.float64)
    loc_mad = (
        1.4826
        * pd.Series(np.abs(e - loc_med))
        .rolling(window=w_loc, min_periods=1, center=True)
        .median()
        .to_numpy(dtype=np.float64)
    )
    eff_mad = np.maximum(loc_mad, np.maximum(0.5 * glob_mad, eps))
    z_loc = np.maximum(0.0, (e - loc_med) / eff_mad)
    z_glob = np.maximum(0.0, (e - glob_med) / glob_mad)
    return 0.5 * (z_loc + z_glob)


def _estimate_window_instability(x: np.ndarray, vol_win: int = 25, eps: float = 1e-6) -> float:
    n = len(x)
    if n < 10:
        return 1.0
    w_v = max(3, min(int(vol_win), n))
    med = float(np.median(x))
    abs_dev = np.abs(x - med)
    glob_mad = max(1.4826 * float(np.median(abs_dev)), eps)
    loc_mad = (
        1.4826
        * pd.Series(abs_dev).rolling(window=w_v, min_periods=1, center=True).median().to_numpy(dtype=np.float64)
    )
    q85_loc_ratio = float(np.quantile(loc_mad, 0.85)) / glob_mad

    diff1 = np.abs(np.diff(x)) if n > 1 else np.zeros(1, dtype=np.float64)
    _, mad_d = _mad_scale(diff1, eps=eps)
    high_grad_ratio = float(np.mean(diff1 > 3.0 * mad_d))

    instab = 0.65 * max(0.0, q85_loc_ratio - 1.0) + 1.50 * max(0.0, high_grad_ratio - 0.05)
    return float(np.clip(1.0 + instab, 1.0, 2.5))


def _compute_regime_change_gate(
    x: np.ndarray,
    volt_win: Optional[np.ndarray],
    cfg: Config,
    eps: float = 1e-6,
) -> np.ndarray:
    n = len(x)
    gate = np.ones(n, dtype=np.float64)
    strength = float(getattr(cfg, "regime_gate_strength", 0.25))
    if n < 10 or strength <= 0.0:
        return gate

    s_x = pd.Series(x, dtype=np.float64)
    trend_x = s_x.rolling(window=min(21, n), min_periods=1, center=True).median().to_numpy(dtype=np.float64)
    dx = np.zeros(n, dtype=np.float64)
    dx[1:] = np.abs(np.diff(trend_x))
    _, sc_dx = _mad_scale(dx, eps=eps)
    z_dx = np.clip(dx / sc_dx, 0.0, 4.0) / 4.0

    if volt_win is not None and len(volt_win) == n:
        v = _sanitize_1d(volt_win)
        trend_v = pd.Series(v, dtype=np.float64).rolling(window=min(21, n), min_periods=1, center=True).median().to_numpy(dtype=np.float64)
        dv = np.zeros(n, dtype=np.float64)
        dv[1:] = np.abs(np.diff(trend_v))
        _, sc_dv = _mad_scale(dv, eps=eps)
        z_dv = np.clip(dv / sc_dv, 0.0, 4.0) / 4.0
        regime_shift = np.sqrt(0.5 * (z_dx * z_dx) + 0.5 * (z_dv * z_dv))
    else:
        regime_shift = z_dx

    med_x = s_x.rolling(window=5, min_periods=1, center=True).median().to_numpy(dtype=np.float64)
    _, sc_x = _mad_scale(x - med_x, eps=eps)
    spike_ratio = np.clip(np.abs(x - med_x) / (5.0 * sc_x), 0.0, 1.0)

    eff_shift = regime_shift * (1.0 - 0.85 * spike_ratio)
    return 1.0 + strength * eff_shift


def _detect_isolation_forest(win: np.ndarray, cfg: Config) -> Tuple[np.ndarray, np.ndarray, float]:
    n = len(win)
    if n < 15:
        return np.zeros(n, dtype=np.float64), np.zeros(n, dtype=np.float64), 0.0
    s = pd.Series(win, dtype=np.float64)
    d1 = np.zeros(n, dtype=np.float64)
    d1[1:] = np.diff(win)
    roll_med = s.rolling(window=min(15, n), min_periods=1, center=True).median().to_numpy(dtype=np.float64)
    feats = _sanitize_1d(np.column_stack([win, d1, win - roll_med])).reshape(n, 3)
    feats_sc = (feats - np.median(feats, axis=0)) / np.maximum(
        1.4826 * np.median(np.abs(feats - np.median(feats, axis=0)), axis=0), 1e-5
    )
    feats_sc = _sanitize_1d(feats_sc).reshape(n, 3)

    iso = IsolationForest(
        n_estimators=int(max(30, cfg.if_estimators)),
        contamination=float(np.clip(cfg.if_contamination, 0.01, 0.25)),
        random_state=cfg.seed,
        n_jobs=1,
    )
    preds = iso.fit_predict(feats_sc)
    raw_norm = _sanitize_1d(-iso.decision_function(feats_sc))
    med_n, sc_n = _mad_scale(raw_norm)
    level = np.maximum(0.0, (raw_norm - med_n) / sc_n)
    anom_mask = preds == -1
    scores = anom_mask.astype(np.float64) * np.maximum(level, 1.0)
    thr = float(np.min(level[anom_mask])) if np.any(anom_mask) else float(np.quantile(level, 0.94))
    return level, scores, thr


def compute_local_volatility_corridor(
    win: np.ndarray,
    cfg: Config,
    eps: float = 1e-6,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    x = _sanitize_1d(win)
    n = len(x)
    if n == 0:
        empty = np.zeros(0, dtype=np.float64)
        return empty, empty, empty, float(cfg.corridor_k_base)

    s = pd.Series(x)
    w_c = max(1, 2 * max(1, int(cfg.corridor_center_radius)) + 1)
    w_v = max(3, min(int(cfg.corridor_vol_win), n))

    center_t = s.rolling(window=w_c, min_periods=1, center=True).median().to_numpy(dtype=np.float64)
    resid = x - center_t
    abs_res = np.abs(resid)

    loc_mad = (
        1.4826
        * pd.Series(abs_res)
        .rolling(window=w_v, min_periods=1, center=True)
        .median()
        .to_numpy(dtype=np.float64)
    )
    _, glob_mad_res = _mad_scale(resid, eps=eps)
    sigma_floor = max(float(cfg.corridor_sigma_floor_ratio) * glob_mad_res, eps)
    sigma_loc = np.maximum(loc_mad, sigma_floor)

    q50 = max(float(np.quantile(abs_res, 0.50)), eps)
    q95 = float(np.quantile(abs_res, 0.95))
    tail_ratio = q95 / q50

    k_lo, k_hi = cfg.corridor_k_bounds
    k_w = float(
        np.clip(
            float(cfg.corridor_k_base) * (1.0 + float(cfg.corridor_tail_alpha) * max(0.0, tail_ratio - 1.0)),
            k_lo,
            k_hi,
        )
    )

    lower_t = center_t - k_w * sigma_loc
    upper_t = center_t + k_w * sigma_loc

    out_mag = np.maximum(np.maximum(0.0, lower_t - x), np.maximum(0.0, x - upper_t))
    violation_level = out_mag / sigma_loc
    return lower_t, upper_t, violation_level, k_w


def detect_corridor_consensus(
    win: np.ndarray,
    active_base_scores: Dict[str, np.ndarray],
    cfg: Config,
    lstm_errs: Optional[np.ndarray] = None,
    volt_win: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray, float]:
    x = _sanitize_1d(win)
    n = len(x)
    instab_idx = _estimate_window_instability(x, vol_win=cfg.corridor_vol_win)
    instab_mult = 1.0 + float(getattr(cfg, "consensus_instab_slope", 0.25)) * max(0.0, instab_idx - 1.0)
    base_cons_thr = float(getattr(cfg, "consensus_level_thresh", 1.15)) * instab_mult

    if n == 0 or not active_base_scores:
        return np.zeros(0, dtype=np.float64), np.zeros(0, dtype=np.float64), base_cons_thr

    regime_gate = _compute_regime_change_gate(x, volt_win, cfg)

    n_methods = len(active_base_scores)
    votes = np.zeros(n, dtype=np.int32)
    for sc_arr in active_base_scores.values():
        votes += (_sanitize_1d(sc_arr) > 0.0).astype(np.int32)

    req_votes = max(1, int(np.ceil(float(cfg.consensus_min_vote_ratio) * n_methods)))
    req_votes = max(int(cfg.consensus_min_votes), req_votes) if n_methods >= int(cfg.consensus_min_votes) else 1
    stage1_pass = votes >= req_votes

    _, _, viol_lvl, k_w = compute_local_volatility_corridor(x, cfg)
    corridor_norm = viol_lvl / max(k_w, 1e-6)

    w_c = max(1, 2 * max(1, int(cfg.corridor_center_radius)) + 1)
    med_center = pd.Series(x).rolling(window=w_c, min_periods=1, center=True).median().to_numpy(dtype=np.float64)
    _, glob_mad_x = _mad_scale(x - med_center)
    impulse_z = np.abs(x - med_center) / glob_mad_x

    z_thr = float(cfg.consensus_lstm_z_thresh) * (1.0 + 0.15 * max(0.0, instab_idx - 1.0))
    if lstm_errs is not None and len(lstm_errs) == n and np.any(lstm_errs > 0.0):
        z_lstm = _robust_z_from_errs(lstm_errs, local_win=cfg.corridor_vol_win)
        seq_cut = min(n, int(cfg.lstm_seq_len))
        if seq_cut > 0:
            z_lstm[:seq_cut] = impulse_z[:seq_cut]
        eff_dyn_z = np.maximum(z_lstm, 0.85 * impulse_z)
        lstm_norm = eff_dyn_z / max(z_thr, 1e-6)
    else:
        lstm_norm = impulse_z / max(z_thr, 1e-6)

    dual_cut = float(cfg.consensus_dual_norm_thresh) * instab_mult * regime_gate
    strong_corr_cut = float(cfg.consensus_strong_corridor_norm) * instab_mult * regime_gate
    strong_lstm_cut = float(cfg.consensus_strong_lstm_norm) * instab_mult * regime_gate

    dual_match = (corridor_norm >= dual_cut) & (lstm_norm >= dual_cut)
    strong_impulse = corridor_norm >= strong_corr_cut
    strong_dynamics = lstm_norm >= strong_lstm_cut
    stage2_pass = dual_match | strong_impulse | strong_dynamics

    vote_ratio = votes / max(1.0, float(n_methods))
    blended_dev = 0.55 * corridor_norm + 0.45 * lstm_norm
    peak_dev = 0.80 * np.maximum(corridor_norm, lstm_norm)
    combined_level = np.maximum(blended_dev, peak_dev) * (0.65 + 0.35 * vote_ratio)

    eff_cons_thr_vec = base_cons_thr * regime_gate
    confirmed_mask = stage1_pass & stage2_pass & (combined_level >= eff_cons_thr_vec)
    scores = confirmed_mask.astype(np.float64) * combined_level

    return combined_level, scores, float(np.mean(eff_cons_thr_vec))


def detect_adaptive_trend_deriv_var(win: np.ndarray, cfg: Config) -> Tuple[np.ndarray, np.ndarray, float]:
    x = _sanitize_1d(win)
    n = len(x)
    if n == 0:
        return np.zeros(0, dtype=np.float64), np.zeros(0, dtype=np.float64), float(cfg.tdv_mad_mult)

    s = pd.Series(x)
    w_tr = max(3, min(int(cfg.tdv_trend_win), n))
    w_sm = max(1, min(int(cfg.tdv_smooth_win), n))
    w_sl = max(3, min(int(cfg.tdv_slope_win), n))

    rob_trend = (
        s.rolling(window=w_tr, min_periods=1, center=True)
        .median()
        .rolling(window=w_sm, min_periods=1, center=True)
        .mean()
        .to_numpy(dtype=np.float64)
    )
    resid = x - rob_trend
    _, sc_res = _mad_scale(resid)
    z_res = np.abs(resid) / sc_res

    slopes = np.zeros(n, dtype=np.float64)
    for i in range(n):
        lo = max(0, i - w_sl + 1)
        seg = rob_trend[lo : i + 1]
        m_len = len(seg)
        if m_len >= 2:
            tc = np.arange(m_len, dtype=np.float64) - 0.5 * (m_len - 1)
            denom = float(np.sum(tc * tc)) + 1e-12
            slopes[i] = float(np.sum(tc * (seg - seg.mean()))) / denom

    slope_jump = np.zeros(n, dtype=np.float64)
    step_lag = max(1, w_sl // 2)
    if n > step_lag:
        slope_jump[step_lag:] = np.abs(slopes[step_lag:] - slopes[:-step_lag])
    med_sj, sc_sj = _mad_scale(slope_jump)
    z_jump = np.maximum(0.0, (slope_jump - med_sj) / sc_sj)

    w_res, w_jmp = cfg.tdv_weights
    w_sum = max(w_res + w_jmp, 1e-12)
    w_res, w_jmp = w_res / w_sum, w_jmp / w_sum

    level = np.sqrt(w_res * (z_res * z_res) + w_jmp * (z_jump * z_jump))

    lvl_med, lvl_mad = _mad_scale(level)
    q_cut = float(np.quantile(level, np.clip(cfg.tdv_tail_quantile, 0.50, 0.995)))
    thr = float(max(cfg.tdv_mad_mult, min(lvl_med + cfg.tdv_mad_mult * lvl_mad, max(q_cut, cfg.tdv_mad_mult))))

    cands = np.where(level > thr)[0]
    scores = np.zeros(n, dtype=np.float64)
    if len(cands) > 0:
        order = cands[np.argsort(-level[cands])]
        suppressed = np.zeros(n, dtype=bool)
        r_nms = max(0, int(cfg.tdv_nms_radius))
        for idx in order:
            if not suppressed[idx]:
                scores[idx] = level[idx]
                if r_nms > 0:
                    suppressed[max(0, idx - r_nms) : min(n, idx + r_nms + 1)] = True

    return level, scores, thr


def detect_robust_dyn_dur(
    win: np.ndarray,
    cfg: Config,
    lstm_errs: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray, float]:
    n = len(win)
    if n == 0:
        return np.zeros(0, dtype=np.float64), np.zeros(0, dtype=np.float64), 1.0

    eps = 1e-6
    s = pd.Series(win, dtype=np.float64)
    smooth_win = int(np.clip(round(0.01 * n), 3, 15))
    var_win = int(np.clip(round(0.04 * n), 10, 40))
    min_dur = int(np.clip(round(float(cfg.dyn_min_dur)), 3, 12))

    smooth_sig = s.rolling(window=smooth_win, min_periods=1).mean().to_numpy(dtype=np.float64)
    med_s = float(np.median(smooth_sig))
    mad_s = float(np.median(np.abs(smooth_sig - med_s))) * 1.4826 + 1e-4

    roll_mean = s.rolling(window=var_win, min_periods=1).mean().to_numpy(dtype=np.float64)
    norm_lvl = (np.abs(win - roll_mean) / mad_s) / max(float(cfg.dyn_lvl_thresh), 1e-4)

    deriv = np.zeros(n, dtype=np.float64)
    if n > 1:
        deriv[1:] = np.diff(smooth_sig)
    med_d = float(np.median(deriv))
    mad_d = float(np.median(np.abs(deriv - med_d))) * 1.4826 + 1e-4
    norm_slope = (np.abs(deriv - med_d) / mad_d) / max(float(cfg.dyn_slope_thresh), 1e-4)

    loc_var = s.rolling(window=var_win, min_periods=1).var(ddof=0).fillna(0.0).to_numpy(dtype=np.float64)
    med_v = float(np.median(loc_var))
    var_scale = max(med_v + 1.4826 * float(np.median(np.abs(loc_var - med_v))), 1e-4)
    norm_var = (loc_var / var_scale) / max(float(cfg.dyn_var_thresh), 1e-4)

    d_shifted = deriv - med_d
    slope_cut = float(cfg.dyn_slope_k) * mad_d

    def _run_len_1d(mask_1d: np.ndarray) -> np.ndarray:
        cs = np.cumsum(mask_1d.astype(np.int32))
        reset = np.where(~mask_1d, cs, 0)
        return cs - np.maximum.accumulate(reset)

    max_run = np.maximum(_run_len_1d(d_shifted > slope_cut), _run_len_1d(d_shifted < -slope_cut))
    excess_ratio = np.clip((max_run - min_dur) / float(min_dur), 0.0, 1.0)
    persistence_factor = 1.0 + float(cfg.dyn_pers_alpha) * excess_ratio

    b_lo, b_hi = cfg.dyn_boost_bounds
    level_boost = 0.5 * (b_lo + b_hi)
    base_norm = np.sqrt(0.35 * (norm_lvl * norm_lvl) + 0.40 * (norm_slope * norm_slope) + 0.25 * (norm_var * norm_var))
    lvl = base_norm * persistence_factor * level_boost

    active_components = (
        (norm_lvl > 1.0).astype(np.float64)
        + (norm_slope > 1.0).astype(np.float64)
        + (norm_var > 1.0).astype(np.float64)
    )
    lvl = lvl * (1.0 + cfg.dyn_joint_bonus * np.maximum(0.0, active_components - 1.0)) * (1.0 + cfg.dyn_pers_bonus * excess_ratio)

    if lstm_errs is not None and len(lstm_errs) == n and cfg.dyn_lstm_weight > 0.0:
        err_arr = np.asarray(lstm_errs, dtype=np.float64)
        _, mad_e = _mad_scale(err_arr, eps=eps)
        lvl = lvl * (1.0 + cfg.dyn_lstm_weight * np.clip(err_arr / mad_e, 0.0, 4.0))

    lvl_med, lvl_mad = _mad_scale(lvl, eps=eps)
    _, q_hi = cfg.dyn_peak_quantile_bounds
    thr = float(max(lvl_med + cfg.dyn_peak_mad_k * lvl_mad, float(np.quantile(lvl, q_hi))))
    scores = (lvl > thr).astype(np.float64) * lvl
    return lvl, scores, thr


def detect_enabled_methods(
    win: np.ndarray,
    lstm_errs: np.ndarray,
    cfg: Config,
    cal_errs: Optional[np.ndarray] = None,
    rm_state_thresh: Optional[float] = None,
    aci_state_alpha: Optional[float] = None,
    volt_win: Optional[np.ndarray] = None,
) -> Tuple[Dict[str, Tuple[np.ndarray, np.ndarray, float]], float, float]:
    win_clean = _sanitize_1d(win)
    errs_clean = _sanitize_1d(lstm_errs)
    w_len = len(win_clean)
    med = float(np.median(win_clean))
    mad = float(np.median(np.abs(win_clean - med))) * 1.4826 + 1e-4
    p = min(cfg.ar_order, max(1, w_len // 10))

    mad_level = np.abs(win_clean - med) / mad
    m1_scores = (mad_level > cfg.mad_k).astype(np.float64) * mad_level

    ar_err = np.zeros(w_len, dtype=np.float64)
    if w_len > p + 5 and cfg.enabled_methods.get("AR", False):
        X_ar, y_ar = _make_lag_matrix(win_clean, p)
        ar_model = Ridge(alpha=1.0).fit(X_ar, y_ar)
        ar_err[p:] = np.abs(y_ar - ar_model.predict(X_ar))
    ar_scale = float(np.median(ar_err) + np.median(np.abs(ar_err - np.median(ar_err))) * 1.4826) + 1e-4
    ar_level = ar_err / ar_scale
    m3_scores = (ar_level > cfg.ar_thresh).astype(np.float64) * ar_level

    q_hi = float(np.quantile(win_clean, cfg.quantile_q))
    q_lo = float(np.quantile(win_clean, 1.0 - cfg.quantile_q))
    span = max(q_hi - q_lo, 1e-4)
    q_level = (np.maximum(0.0, win_clean - q_hi) + np.maximum(0.0, q_lo - win_clean)) / span
    m6_scores = (q_level > 0.0).astype(np.float64) * (1.0 + q_level * 10.0)

    peak_src = 0.5 * mad_level + 0.5 * ar_level
    nms_thresh = float(np.quantile(peak_src, cfg.nms_quantile))
    m8_scores = _nms_1d(peak_src, radius=cfg.nms_radius, thresh_quantile=cfg.nms_quantile)

    dyn_lvl, dyn_sc, dyn_thr = detect_robust_dyn_dur(win_clean, cfg, lstm_errs=errs_clean)
    tdv_lvl, tdv_sc, tdv_thr = detect_adaptive_trend_deriv_var(win_clean, cfg)
    if_lvl, if_sc, if_thr = (
        _detect_isolation_forest(win_clean, cfg)
        if cfg.enabled_methods.get(IF_METHOD_NAME, False)
        else (np.zeros(w_len, dtype=np.float64), np.zeros(w_len, dtype=np.float64), 0.0)
    )

    all_base_map: Dict[str, Tuple[np.ndarray, np.ndarray, float]] = {
        "Median / MAD": (mad_level, m1_scores, cfg.mad_k),
        "AR": (ar_level, m3_scores, cfg.ar_thresh),
        "Quantile Thresholding": (q_level, m6_scores, 0.0),
        "NMS": (peak_src, m8_scores, nms_thresh),
        DYN_METHOD_NAME: (dyn_lvl, dyn_sc, dyn_thr),
        TDV_METHOD_NAME: (tdv_lvl, tdv_sc, tdv_thr),
        IF_METHOD_NAME: (if_lvl, if_sc, if_thr),
    }

    active_base = [m for m in get_active_base_methods(cfg) if m in all_base_map]
    active_out: Dict[str, Tuple[np.ndarray, np.ndarray, float]] = {m: all_base_map[m] for m in active_base}

    if cfg.enabled_methods.get(CONSENSUS_METHOD_NAME, False) and len(active_base) > 0:
        base_sc_only = {m: active_out[m][1] for m in active_base}
        cons_lvl, cons_sc, cons_thr = detect_corridor_consensus(
            win_clean, base_sc_only, cfg, lstm_errs=errs_clean, volt_win=volt_win
        )
        active_out[CONSENSUS_METHOD_NAME] = (cons_lvl, cons_sc, cons_thr)

    next_rm = cfg.rm_init_thresh if rm_state_thresh is None else rm_state_thresh
    next_aci = cfg.aci_base_alpha if aci_state_alpha is None else aci_state_alpha
    return active_out, next_rm, next_aci


# ==============================================================================
# 6. SCENARIO-SEPARATED CONTROL VALIDATION, SUMMARIES & WILCOXON-HOLM TEST
# ==============================================================================
def _prepare_independent_clean_window(raw_win: np.ndarray, mad_k_cut: float) -> np.ndarray:
    x = _sanitize_1d(raw_win).copy()
    n = len(x)
    if n < 5:
        return x
    s = pd.Series(x)
    loc_med = s.rolling(window=11, min_periods=1, center=True).median().to_numpy(dtype=np.float64)
    _, mad_res = _mad_scale(x - loc_med)
    outlier_mask = np.abs(x - loc_med) > float(mad_k_cut) * mad_res
    if np.any(outlier_mask):
        x_clean = x.copy()
        x_clean[outlier_mask] = np.nan
        return _sanitize_1d(pd.Series(x_clean).interpolate(method="linear").ffill().bfill().to_numpy(dtype=np.float64))
    return x


def _select_temporal_checkpoint(
    w_start: int,
    checkpoints_by_start: Dict[int, Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]],
    fallback_raw_sd: Dict[str, torch.Tensor],
    fallback_rep_sd: Dict[str, torch.Tensor],
) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]:
    if not checkpoints_by_start:
        return fallback_raw_sd, fallback_rep_sd
    avail_starts = sorted(checkpoints_by_start.keys())
    past_or_eq = [s for s in avail_starts if s <= w_start]
    chosen_s = past_or_eq[-1] if past_or_eq else avail_starts[0]
    return checkpoints_by_start[chosen_s]


def _compute_event_metrics(
    pred_mask: np.ndarray,
    gt_mask: np.ndarray,
    tolerance: int = 5,
) -> Tuple[float, float, float]:
    n = len(gt_mask)
    if n == 0 or not np.any(gt_mask):
        return 0.0, 0.0, 0.0

    events: List[Tuple[int, int]] = []
    i = 0
    while i < n:
        if not gt_mask[i]:
            i += 1
            continue
        st = i
        while i < n and gt_mask[i]:
            i += 1
        events.append((st, i - 1))

    tol = max(0, int(tolerance))
    ev_tp = 0
    for st, en in events:
        lo = max(0, st - tol)
        hi = min(n, en + tol + 1)
        if np.any(pred_mask[lo:hi]):
            ev_tp += 1

    ev_rec = ev_tp / max(1, len(events))

    pred_clusters: List[Tuple[int, int]] = []
    j = 0
    while j < n:
        if not pred_mask[j]:
            j += 1
            continue
        st = j
        while j < n and pred_mask[j]:
            j += 1
        pred_clusters.append((st, j - 1))

    if not pred_clusters:
        return 0.0, float(ev_rec), 0.0

    ev_fp = 0
    for pst, pen in pred_clusters:
        matched = any(
            (pen >= max(0, gst - tol)) and (pst <= min(n - 1, gen + tol))
            for gst, gen in events
        )
        if not matched:
            ev_fp += 1

    ev_prec = (len(pred_clusters) - ev_fp) / max(1, len(pred_clusters))
    ev_f1 = (2.0 * ev_prec * ev_rec / (ev_prec + ev_rec)) if (ev_prec + ev_rec) > 0.0 else 0.0
    return float(ev_prec), float(ev_rec), float(ev_f1)


def _eval_detector_on_scenario(
    scenario_name: str,
    seed: int,
    client_id: int,
    w_start: int,
    det_pos: Dict[str, Tuple[np.ndarray, np.ndarray, float]],
    det_neg: Dict[str, Tuple[np.ndarray, np.ndarray, float]],
    gt_mask: np.ndarray,
    tolerance: int,
) -> List[Dict[str, object]]:
    out: List[Dict[str, object]] = []
    for m_name in det_pos:
        _, sc_pos, _ = det_pos[m_name]
        _, sc_neg, _ = det_neg[m_name]
        pred_pos_mask = sc_pos > 0.0
        pred_neg_mask = sc_neg > 0.0

        tp = int(np.sum(pred_pos_mask & gt_mask))
        fp = int(np.sum(pred_pos_mask & (~gt_mask)))
        fn = int(np.sum((~pred_pos_mask) & gt_mask))
        prec = tp / max(1, tp + fp)
        rec = tp / max(1, tp + fn)
        f1 = (2.0 * prec * rec / (prec + rec)) if (prec + rec) > 0.0 else 0.0
        ev_prec, ev_rec, ev_f1 = _compute_event_metrics(pred_pos_mask, gt_mask, tolerance=tolerance)
        fpr_neg = float(np.mean(pred_neg_mask))

        out.append(
            {
                "seed": int(seed),
                "client": int(client_id),
                "win_start": int(w_start),
                "scenario": scenario_name,
                "method": m_name,
                "ctrl_tp": tp,
                "ctrl_fp": fp,
                "ctrl_fn": fn,
                "ctrl_precision": round(prec, 4),
                "ctrl_recall": round(rec, 4),
                "ctrl_f1": round(f1, 4),
                "ctrl_event_precision": round(ev_prec, 4),
                "ctrl_event_recall": round(ev_rec, 4),
                "ctrl_event_f1": round(ev_f1, 4),
                "neg_ctrl_fpr": round(fpr_neg, 4),
            }
        )
    return out


def run_controlled_anomaly_validation(
    raw_signals: List[np.ndarray],
    voltage_signals: List[np.ndarray],
    checkpoints_by_client_and_start: List[Dict[int, Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]]],
    fallback_raw_sds: List[Dict[str, torch.Tensor]],
    fallback_rep_sds: List[Dict[str, torch.Tensor]],
    cfg: Config,
) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    rng = np.random.default_rng(cfg.seed + 777)
    eval_model = _build_forecast_model(cfg)

    for c_idx, sig in enumerate(raw_signals):
        total_n = len(sig)
        if total_n < cfg.window_size:
            continue

        volt_sig = voltage_signals[c_idx] if c_idx < len(voltage_signals) else None
        max_start = total_n - cfg.window_size
        starts = sorted(
            {
                min(max_start, max(0, int(w_i) * cfg.step_size))
                for w_i in cfg.ctrl_window_indices
            }
        )
        ckpt_map = checkpoints_by_client_and_start[c_idx]
        fb_raw = fallback_raw_sds[c_idx]
        fb_rep = fallback_rep_sds[c_idx]

        for w_start in starts:
            raw_net_sd, _ = _select_temporal_checkpoint(w_start, ckpt_map, fb_raw, fb_rep)
            raw_sub = sig[w_start : w_start + cfg.window_size]
            volt_sub = volt_sig[w_start : w_start + cfg.window_size] if volt_sig is not None else None
            base_win = _prepare_independent_clean_window(raw_sub, cfg.ctrl_base_clean_mad_k)
            _, mad_b = _mad_scale(base_win)

            neg_win = _sanitize_1d(base_win + rng.normal(0.0, 0.20 * mad_b, size=cfg.window_size))
            err_neg_pre, _, _ = lstm_window_predict_and_metrics(
                neg_win, raw_net_sd, cfg, eval_target=base_win, reusable_model=eval_model
            )
            det_neg, _, _ = detect_enabled_methods(neg_win, err_neg_pre, cfg, volt_win=volt_sub)

            spike_win = base_win.copy()
            spike_gt = np.zeros(cfg.window_size, dtype=bool)
            cand_positions = np.linspace(
                cfg.lstm_seq_len + 15,
                cfg.window_size - 15,
                num=max(1, int(cfg.ctrl_n_spikes)),
                dtype=int,
            )
            for k_i, pos in enumerate(cand_positions):
                sign = 1.0 if (k_i % 2 == 0) else -1.0
                spike_win[pos] += sign * float(cfg.ctrl_spike_sigma_mult) * mad_b
                spike_gt[pos] = True
            spike_win = _sanitize_1d(spike_win)
            err_sp_pre, _, _ = lstm_window_predict_and_metrics(
                spike_win, raw_net_sd, cfg, eval_target=base_win, reusable_model=eval_model
            )
            det_sp, _, _ = detect_enabled_methods(spike_win, err_sp_pre, cfg, volt_win=volt_sub)
            rows.extend(
                _eval_detector_on_scenario(
                    "spikes_only", cfg.seed, c_idx + 1, w_start, det_sp, det_neg, spike_gt, cfg.nms_event_tolerance
                )
            )

            if cfg.ctrl_include_zero_out_scenario and cfg.window_size > cfg.lstm_seq_len + 40:
                zero_win = base_win.copy()
                zero_gt = np.zeros(cfg.window_size, dtype=bool)
                z_lo = cfg.window_size // 2
                z_hi = min(cfg.window_size - 10, z_lo + 8)
                zero_win[z_lo:z_hi] = 0.0
                zero_gt[z_lo:z_hi] = True
                zero_win = _sanitize_1d(zero_win)
                err_zr_pre, _, _ = lstm_window_predict_and_metrics(
                    zero_win, raw_net_sd, cfg, eval_target=base_win, reusable_model=eval_model
                )
                det_zr, _, _ = detect_enabled_methods(zero_win, err_zr_pre, cfg, volt_win=volt_sub)
                rows.extend(
                    _eval_detector_on_scenario(
                        "zero_out_segment", cfg.seed, c_idx + 1, w_start, det_zr, det_neg, zero_gt, cfg.nms_event_tolerance
                    )
                )

    return pd.DataFrame(rows)


def summarize_control_methods(ctrl_all_df: pd.DataFrame) -> pd.DataFrame:
    if ctrl_all_df.empty:
        return pd.DataFrame()
    return (
        ctrl_all_df.groupby(["fl_mode", "method"], as_index=False)
        .agg(
            n_eval_windows=("win_start", "count"),
            mean_precision=("ctrl_precision", "mean"),
            std_precision=("ctrl_precision", "std"),
            mean_recall=("ctrl_recall", "mean"),
            std_recall=("ctrl_recall", "std"),
            mean_f1=("ctrl_f1", "mean"),
            std_f1=("ctrl_f1", "std"),
            mean_event_f1=("ctrl_event_f1", "mean"),
            mean_event_recall=("ctrl_event_recall", "mean"),
            mean_neg_fpr=("neg_ctrl_fpr", "mean"),
            std_neg_fpr=("neg_ctrl_fpr", "std"),
        )
        .sort_values(["fl_mode", "mean_f1"], ascending=[True, False])
    )


def summarize_control_by_scenario(ctrl_all_df: pd.DataFrame) -> pd.DataFrame:
    if ctrl_all_df.empty or "scenario" not in ctrl_all_df.columns:
        return pd.DataFrame()
    return (
        ctrl_all_df.groupby(["fl_mode", "scenario", "method"], as_index=False)
        .agg(
            mean_precision=("ctrl_precision", "mean"),
            mean_recall=("ctrl_recall", "mean"),
            mean_f1=("ctrl_f1", "mean"),
            mean_event_f1=("ctrl_event_f1", "mean"),
            mean_event_recall=("ctrl_event_recall", "mean"),
            mean_neg_fpr=("neg_ctrl_fpr", "mean"),
        )
        .sort_values(["fl_mode", "scenario", "mean_event_f1"], ascending=[True, True, False])
    )


def summarize_high_anomaly_oos(lstm_recs: List[Dict[str, object]], mode: str) -> pd.DataFrame:
    if not lstm_recs:
        return pd.DataFrame()
    df = pd.DataFrame(lstm_recs)
    if "is_prequential_oos" in df.columns:
        df = df[df["is_prequential_oos"] == True]
    if df.empty:
        return pd.DataFrame()

    rows: List[Dict[str, object]] = []
    for c_id in sorted(df["client"].unique()):
        sub = df[df["client"] == c_id]
        med_rep = float(sub["repaired_points"].median())
        hi_sub = sub[sub["repaired_points"] >= med_rep]
        if hi_sub.empty:
            hi_sub = sub
        m1_mse = float(hi_sub["raw_net_mse"].mean())
        m2_mse = float(hi_sub["rep_net_mse"].mean())
        m3_mse = float(hi_sub["fedavg_net3_mse"].mean()) if "fedavg_net3_mse" in hi_sub.columns else 0.0
        pers_mse = float(hi_sub["pers_clean_mse"].mean()) if "pers_clean_mse" in hi_sub.columns else 0.0
        m1_r2 = float(hi_sub["raw_net_r2"].mean())
        m2_r2 = float(hi_sub["rep_net_r2"].mean())
        m3_r2 = float(hi_sub["fedavg_net3_r2"].mean()) if "fedavg_net3_r2" in hi_sub.columns else 0.0
        impr_pct = ((m1_mse - m2_mse) / max(abs(m1_mse), 1e-12)) * 100.0
        impr3_pct = ((m1_mse - m3_mse) / max(abs(m1_mse), 1e-12)) * 100.0
        rows.append(
            {
                "fl_mode": mode,
                "client": int(c_id),
                "hi_anom_records": int(len(hi_sub)),
                "min_rep_pts_cut": round(med_rep, 1),
                "pers_hi_mse": round(pers_mse, 6),
                "net1_hi_mse": round(m1_mse, 6),
                "net2_hi_mse": round(m2_mse, 6),
                "net3_fedavg_hi_mse": round(m3_mse, 6),
                "hi_mse_impr_pct": round(impr_pct, 2),
                "net3_hi_mse_impr_pct": round(impr3_pct, 2),
                "net1_hi_r2": round(m1_r2, 4),
                "net2_hi_r2": round(m2_r2, 4),
                "net3_fedavg_hi_r2": round(m3_r2, 4),
            }
        )
    return pd.DataFrame(rows)


def build_methods_effectiveness_matrix(
    ctrl_sum_df: pd.DataFrame,
    stream_counts_rows: List[Dict[str, object]],
    n_total: int,
) -> pd.DataFrame:
    if ctrl_sum_df.empty:
        return pd.DataFrame()

    stream_map: Dict[Tuple[str, str], List[int]] = {}
    for row in stream_counts_rows:
        mode = str(row.get("fl_mode", ""))
        for k, v in row.items():
            if k not in (
                "fl_mode",
                "client",
                "consensus_anomalies",
                "distinct_anomalies_core",
                "distinct_anomalies_all",
            ):
                stream_map.setdefault((mode, k), []).append(int(v))

    out_rows: List[Dict[str, object]] = []
    for _, r in ctrl_sum_df.iterrows():
        mode = str(r["fl_mode"])
        m_name = str(r["method"])
        vals = stream_map.get((mode, m_name), [])
        mean_pts = float(np.mean(vals)) if vals else 0.0
        pct_series = (mean_pts / max(1.0, float(n_total))) * 100.0
        out_rows.append(
            {
                "fl_mode": mode,
                "method": m_name,
                "is_core_robust": int(m_name in CORE_ROBUST_METHODS),
                "mean_precision": round(float(r["mean_precision"]), 4),
                "mean_recall": round(float(r["mean_recall"]), 4),
                "mean_f1": round(float(r["mean_f1"]), 4),
                "mean_event_f1": round(float(r["mean_event_f1"]), 4) if "mean_event_f1" in r else 0.0,
                "mean_event_recall": round(float(r["mean_event_recall"]), 4) if "mean_event_recall" in r else 0.0,
                "mean_neg_fpr": round(float(r["mean_neg_fpr"]), 4),
                "stream_mean_pts": round(mean_pts, 1),
                "stream_pct_of_series": round(pct_series, 2),
            }
        )
    return pd.DataFrame(out_rows).sort_values(["fl_mode", "mean_f1"], ascending=[True, False])


def build_fl_modes_comparison_summary(
    all_oos_frames: List[pd.DataFrame],
    all_hi_frames: List[pd.DataFrame],
    w_ms_df: pd.DataFrame,
) -> pd.DataFrame:
    if not all_oos_frames:
        return pd.DataFrame()
    full_oos = pd.concat(all_oos_frames, ignore_index=True)
    if full_oos.empty:
        return pd.DataFrame()

    hi_cat = pd.concat(all_hi_frames, ignore_index=True) if all_hi_frames else pd.DataFrame()
    hi_lookup: Dict[Tuple[str, int], Dict[str, float]] = {}
    if not hi_cat.empty:
        for _, hr in hi_cat.iterrows():
            hi_lookup[(str(hr["fl_mode"]), int(hr["client"]))] = {
                "hi_mse_impr_pct": float(hr["hi_mse_impr_pct"]),
                "net2_hi_r2": float(hr["net2_hi_r2"]),
            }

    w_lookup: Dict[Tuple[str, int], Tuple[bool, float]] = {}
    if not w_ms_df.empty:
        w_mse = w_ms_df[w_ms_df["metric"] == "mse"]
        for _, wr in w_mse.iterrows():
            w_lookup[(str(wr["fl_mode"]), int(wr["client"]))] = (
                bool(wr["all_significant_holm_lt_0_01"]),
                float(wr["max_p_value_holm"]),
            )

    base_agg = (
        full_oos.groupby(["fl_mode", "fl_exchange", "client"], as_index=False)
        .agg(
            pers_clean_mse=("pers_clean_mse", "mean"),
            net1_oos_mse=("raw_net_mse", "mean"),
            net2_oos_mse=("rep_net_mse", "mean"),
            net3_fedavg_oos_mse=("fedavg_net3_mse", "mean"),
            net1_on_rep_mse=("net1_on_rep_mse", "mean"),
            net2_on_raw_mse=("net2_on_raw_mse", "mean"),
            pers_all_valid_mse=("pers_all_valid_mse", "mean"),
            net1_all_valid_mse=("raw_all_valid_mse", "mean"),
            net2_all_valid_mse=("rep_all_valid_mse", "mean"),
            net3_all_valid_mse=("fedavg_net3_all_valid_mse", "mean"),
            mean_transition_pts=("transition_pts", "mean"),
            net1_transition_mse=("raw_transition_mse", "mean"),
            net2_transition_mse=("rep_transition_mse", "mean"),
            net3_transition_mse=("fedavg_net3_transition_mse", "mean"),
            net1_oos_r2=("raw_net_r2", "mean"),
            net2_oos_r2=("rep_net_r2", "mean"),
            net3_fedavg_oos_r2=("fedavg_net3_r2", "mean"),
        )
    )

    rows: List[Dict[str, object]] = []
    for _, br in base_agg.iterrows():
        mode = str(br["fl_mode"])
        c_id = int(br["client"])
        pers_c = float(br["pers_clean_mse"])
        m1 = float(br["net1_oos_mse"])
        m2 = float(br["net2_oos_mse"])
        m3 = float(br["net3_fedavg_oos_mse"])
        m1_all = float(br["net1_all_valid_mse"])
        m2_all = float(br["net2_all_valid_mse"])
        m1_tr = float(br["net1_transition_mse"])
        m2_tr = float(br["net2_transition_mse"])
        m3_tr = float(br["net3_transition_mse"])
        impr2_pct = ((m1 - m2) / max(abs(m1), 1e-12)) * 100.0
        impr3_pct = ((m1 - m3) / max(abs(m1), 1e-12)) * 100.0
        all_impr2_pct = ((m1_all - m2_all) / max(abs(m1_all), 1e-12)) * 100.0
        tr_impr2_pct = ((m1_tr - m2_tr) / max(abs(m1_tr), 1e-12)) * 100.0 if (np.isfinite(m1_tr) and np.isfinite(m2_tr)) else 0.0
        tr_impr3_pct = ((m1_tr - m3_tr) / max(abs(m1_tr), 1e-12)) * 100.0 if (np.isfinite(m1_tr) and np.isfinite(m3_tr)) else 0.0
        hi_info = hi_lookup.get((mode, c_id), {"hi_mse_impr_pct": 0.0, "net2_hi_r2": 0.0})
        sig_flag, max_p = w_lookup.get((mode, c_id), (False, 1.0))

        rows.append(
            {
                "fl_mode": mode,
                "fl_exchange": int(br["fl_exchange"]),
                "client": c_id,
                "pers_clean_mse": round(pers_c, 6),
                "net1_oos_mse": round(m1, 6),
                "net2_oos_mse": round(m2, 6),
                "net3_fedavg_oos_mse": round(m3, 6),
                "net1_on_rep_mse": round(float(br["net1_on_rep_mse"]), 6),
                "net2_on_raw_mse": round(float(br["net2_on_raw_mse"]), 6),
                "pers_all_valid_mse": round(float(br["pers_all_valid_mse"]), 6),
                "net1_all_valid_mse": round(m1_all, 6),
                "net2_all_valid_mse": round(m2_all, 6),
                "net3_all_valid_mse": round(float(br["net3_all_valid_mse"]), 6),
                "mean_transition_pts": round(float(br["mean_transition_pts"]), 1),
                "net1_transition_mse": round(m1_tr, 6) if np.isfinite(m1_tr) else float("nan"),
                "net2_transition_mse": round(m2_tr, 6) if np.isfinite(m2_tr) else float("nan"),
                "net3_transition_mse": round(m3_tr, 6) if np.isfinite(m3_tr) else float("nan"),
                "transition_net2_vs_net1_impr_pct": round(tr_impr2_pct, 2),
                "transition_net3_vs_net1_impr_pct": round(tr_impr3_pct, 2),
                "net2_vs_net1_impr_pct": round(impr2_pct, 2),
                "net3_fedavg_vs_net1_impr_pct": round(impr3_pct, 2),
                "all_valid_net2_vs_net1_impr_pct": round(all_impr2_pct, 2),
                "net1_oos_r2": round(float(br["net1_oos_r2"]), 4),
                "net2_oos_r2": round(float(br["net2_oos_r2"]), 4),
                "net3_fedavg_oos_r2": round(float(br["net3_fedavg_oos_r2"]), 4),
                "hi_anom_mse_impr_pct": round(float(hi_info["hi_mse_impr_pct"]), 2),
                "net2_hi_r2": round(float(hi_info["net2_hi_r2"]), 4),
                "wilcoxon_mse_all_sig_holm": sig_flag,
                "max_p_holm_mse": max_p,
            }
        )
    return pd.DataFrame(rows)


def print_real_networks_and_fl_focus(all_oos_frames: List[pd.DataFrame]) -> None:
    if not all_oos_frames:
        return
    full_oos = pd.concat(all_oos_frames, ignore_index=True)
    if full_oos.empty:
        return

    print("\n================ REAL OOS ERRORS: CLEAN, ALL-VALID & STEP-TRANSITION SEGMENTS ================")
    agg_all = (
        full_oos.groupby(["fl_mode", "client"], as_index=False)
        .agg(
            pers_clean_mse=("pers_clean_mse", "mean"),
            net1_clean_mse=("raw_net_mse", "mean"),
            net2_clean_mse=("rep_net_mse", "mean"),
            net3_clean_mse=("fedavg_net3_mse", "mean"),
            net1_all_mse=("raw_all_valid_mse", "mean"),
            net2_all_mse=("rep_all_valid_mse", "mean"),
            net3_all_mse=("fedavg_net3_all_valid_mse", "mean"),
            net1_trans_mse=("raw_transition_mse", "mean"),
            net2_trans_mse=("rep_transition_mse", "mean"),
            net3_trans_mse=("fedavg_net3_transition_mse", "mean"),
        )
    )
    print(agg_all.to_string(index=False))


def _apply_holm_correction(p_values: List[float]) -> List[float]:
    m = len(p_values)
    if m == 0:
        return []
    indexed = sorted(enumerate(p_values), key=lambda x: x[1])
    adjusted = [1.0] * m
    prev = 0.0
    for rank, (orig_idx, p_val) in enumerate(indexed):
        corr = min(1.0, max(prev, (m - rank) * float(p_val)))
        adjusted[orig_idx] = corr
        prev = corr
    return adjusted


def compute_wilcoxon_summary(lstm_recs: List[Dict[str, object]], mode: str) -> pd.DataFrame:
    if not lstm_recs:
        return pd.DataFrame()
    df = pd.DataFrame(lstm_recs)
    if "is_prequential_oos" in df.columns:
        df = df[df["is_prequential_oos"] == True]
    if df.empty:
        return pd.DataFrame()

    out_rows: List[Dict[str, object]] = []
    pairs = [
        ("mse", "raw_net_mse", "rep_net_mse"),
        ("mae", "raw_net_mae", "rep_net_mae"),
        ("r2", "raw_net_r2", "rep_net_r2"),
    ]

    for c_id in sorted(df["client"].unique()):
        sub = df[df["client"] == c_id]
        for metric_label, col1, col2 in pairs:
            v1 = sub[col1].to_numpy(dtype=np.float64)
            v2 = sub[col2].to_numpy(dtype=np.float64)
            valid = np.isfinite(v1) & np.isfinite(v2)
            v1_ok, v2_ok = v1[valid], v2[valid]
            if len(v1_ok) >= 3 and np.any(np.abs(v2_ok - v1_ok) > 1e-12):
                stat, p_val = wilcoxon(v1_ok, v2_ok)
                stat_f, p_f = float(stat), float(p_val)
            else:
                stat_f, p_f = 0.0, 1.0

            if len(v1_ok) > 0:
                med1 = float(np.median(v1_ok))
                med2 = float(np.median(v2_ok))
                iqr1 = float(np.percentile(v1_ok, 75) - np.percentile(v1_ok, 25))
                iqr2 = float(np.percentile(v2_ok, 75) - np.percentile(v2_ok, 25))
                if metric_label in ("mse", "mae"):
                    impr_pct = ((med1 - med2) / max(abs(med1), 1e-12)) * 100.0
                    r2_diff_pp = 0.0
                else:
                    impr_pct = 0.0
                    r2_diff_pp = (med2 - med1) * 100.0
            else:
                med1 = med2 = iqr1 = iqr2 = impr_pct = r2_diff_pp = 0.0

            out_rows.append(
                {
                    "seed": int(sub["seed"].iloc[0]) if "seed" in sub.columns else 0,
                    "fl_mode": mode,
                    "fl_exchange": int(mode_exchanges_params(mode)),
                    "client": int(c_id),
                    "metric": metric_label,
                    "n_pairs": int(len(v1_ok)),
                    "net1_median": round(med1, 6),
                    "net1_iqr": round(iqr1, 6),
                    "net2_median": round(med2, 6),
                    "net2_iqr": round(iqr2, 6),
                    "rel_impr_pct": round(impr_pct, 2),
                    "r2_diff_pp": round(r2_diff_pp, 2),
                    "wilcoxon_stat": round(stat_f, 4),
                    "p_value": p_f,
                    "p_value_holm": p_f,
                    "significant_holm_lt_0_01": False,
                }
            )

    res_df = pd.DataFrame(out_rows)
    if not res_df.empty:
        holm_p = _apply_holm_correction(res_df["p_value"].tolist())
        res_df["p_value_holm"] = holm_p
        res_df["significant_holm_lt_0_01"] = [p < 0.01 for p in holm_p]
    return res_df


def summarize_multiseed_wilcoxon(w_all_df: pd.DataFrame) -> pd.DataFrame:
    if w_all_df.empty:
        return pd.DataFrame()
    return w_all_df.groupby(["fl_mode", "fl_exchange", "client", "metric"], as_index=False).agg(
        n_seeds=("seed", "nunique"),
        mean_net1_median=("net1_median", "mean"),
        mean_net2_median=("net2_median", "mean"),
        mean_rel_impr_pct=("rel_impr_pct", "mean"),
        mean_r2_diff_pp=("r2_diff_pp", "mean"),
        max_p_value_holm=("p_value_holm", "max"),
        all_significant_holm_lt_0_01=("significant_holm_lt_0_01", "all"),
    )


# ==============================================================================
# 7. STREAMING PREQUENTIAL TEST-THEN-TRAIN LOOP WITH THREE LSTM NETWORKS
# ==============================================================================
def run_streaming_fl_and_detection(
    client_signals: List[np.ndarray],
    voltage_signals: List[np.ndarray],
    mode: str,
    cfg: Config,
    txt_log_file,
    csv_writer,
    write_logs: bool = True,
) -> Tuple[
    Dict[int, Dict[str, np.ndarray]],
    Dict[int, Dict[str, np.ndarray]],
    Dict[int, Dict[str, float]],
    List[np.ndarray],
    List[Dict[str, object]],
    List[Dict[int, Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]]],
    List[Dict[str, torch.Tensor]],
    List[Dict[str, torch.Tensor]],
]:
    _set_global_seeds(cfg.seed)
    active_methods = get_active_methods(cfg)
    n_clients = len(client_signals)
    n_total = len(client_signals[0])
    exchange_flag = mode_exchanges_params(mode)

    shared_model = _build_forecast_model(cfg)
    torch.manual_seed(cfg.seed)
    raw_anom_fl_sd = _clone_state_dict_cpu(_build_forecast_model(cfg).state_dict())
    torch.manual_seed(cfg.seed + 1000)
    repaired_fl_sd = _clone_state_dict_cpu(_build_forecast_model(cfg).state_dict())
    torch.manual_seed(cfg.seed + 2000)
    server_fedavg_net3_sd = _clone_state_dict_cpu(_build_forecast_model(cfg).state_dict())

    raw_local_sds = [_clone_state_dict_cpu(raw_anom_fl_sd) for _ in range(n_clients)]
    rep_local_sds = [_clone_state_dict_cpu(repaired_fl_sd) for _ in range(n_clients)]

    checkpoints_per_client: List[Dict[int, Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]]] = [
        {} for _ in range(n_clients)
    ]
    empty_masks = {c: {m: np.zeros(n_total, dtype=bool) for m in active_methods} for c in range(n_clients)}
    empty_levels = {c: {m: np.zeros(n_total, dtype=np.float64) for m in active_methods} for c in range(n_clients)}
    empty_thresh = {c: {m: 0.0 for m in active_methods} for c in range(n_clients)}
    empty_rep_sigs = [np.zeros(n_total, dtype=np.float64) for _ in range(n_clients)]
    if n_total < cfg.window_size or len(active_methods) == 0:
        return (
            empty_masks,
            empty_levels,
            empty_thresh,
            empty_rep_sigs,
            [],
            checkpoints_per_client,
            raw_local_sds,
            rep_local_sds,
        )

    weighted_votes = [
        {m: np.zeros(n_total, dtype=np.float64) for m in active_methods}
        for _ in range(n_clients)
    ]
    weighted_level_num = [
        {m: np.zeros(n_total, dtype=np.float64) for m in active_methods}
        for _ in range(n_clients)
    ]
    total_weights = [
        {m: np.zeros(n_total, dtype=np.float64) for m in active_methods}
        for _ in range(n_clients)
    ]
    thresh_acc = [{m: 0.0 for m in active_methods} for _ in range(n_clients)]
    thresh_cnt = [{m: 0 for m in active_methods} for _ in range(n_clients)]

    rep_num = [np.zeros(n_total, dtype=np.float64) for _ in range(n_clients)]
    rep_den = [np.zeros(n_total, dtype=np.float64) for _ in range(n_clients)]

    buffers = [np.empty(0, dtype=np.float64) for _ in range(n_clients)]
    volt_buffers = [np.empty(0, dtype=np.float64) for _ in range(n_clients)]
    win_starts = [0 for _ in range(n_clients)]
    cal_errs_raw: List[Optional[np.ndarray]] = [None for _ in range(n_clients)]
    rm_state_thresh: List[float] = [cfg.rm_init_thresh for _ in range(n_clients)]
    aci_state_alpha: List[float] = [cfg.aci_base_alpha for _ in range(n_clients)]
    fl_step_idx = 0
    lstm_metrics_records: List[Dict[str, object]] = []

    for chunk_idx, chunk_start in enumerate(range(0, n_total, cfg.stream_chunk_size), start=1):
        chunk_end = min(n_total, chunk_start + cfg.stream_chunk_size)
        for c in range(n_clients):
            buffers[c] = np.concatenate([buffers[c], client_signals[c][chunk_start:chunk_end]])
            if c < len(voltage_signals):
                volt_buffers[c] = np.concatenate([volt_buffers[c], voltage_signals[c][chunk_start:chunk_end]])

        if write_logs and txt_log_file is not None:
            txt_log_file.write(
                f"\n[Chunk #{chunk_idx} | Mode: {mode} | FL_exchange={exchange_flag} | seed={cfg.seed}] "
                f"Indices [{chunk_start}:{chunk_end}) (chunk_size={chunk_end - chunk_start})\n"
            )

        while all(len(buffers[c]) >= cfg.window_size for c in range(n_clients)):
            fl_step_idx += 1
            is_prequential_oos = fl_step_idx > 1
            wins = [_sanitize_1d(buffers[c][: cfg.window_size]).copy() for c in range(n_clients)]
            volt_wins = [
                _sanitize_1d(volt_buffers[c][: cfg.window_size]).copy() if len(volt_buffers[c]) >= cfg.window_size else None
                for c in range(n_clients)
            ]

            for c in range(n_clients):
                ckpt_raw_c = _clone_state_dict_cpu(raw_local_sds[c] if not exchange_flag else raw_anom_fl_sd)
                ckpt_rep_c = _clone_state_dict_cpu(rep_local_sds[c] if not exchange_flag else repaired_fl_sd)
                checkpoints_per_client[c][win_starts[c]] = (ckpt_raw_c, ckpt_rep_c)

            step_det_maps: List[Dict[str, Tuple[np.ndarray, np.ndarray, float]]] = []
            step_rep_wins: List[np.ndarray] = []
            step_rep_counts: List[int] = []
            raw_net_stats: List[Dict[str, float]] = []
            rep_net_stats: List[Dict[str, float]] = []
            net3_srv_stats: List[Dict[str, float]] = []
            pers_stats_list: List[Dict[str, float]] = []
            cross_ablation_list: List[Dict[str, float]] = []

            for c in range(n_clients):
                cur_raw_sd = raw_local_sds[c] if not exchange_flag else raw_anom_fl_sd
                cur_rep_sd = rep_local_sds[c] if not exchange_flag else repaired_fl_sd

                pre_err_c, pre_pred_c, _ = lstm_window_predict_and_metrics(
                    wins[c], cur_raw_sd, cfg, eval_target=wins[c], reusable_model=shared_model
                )

                det_map, next_rm, next_aci = detect_enabled_methods(
                    wins[c],
                    pre_err_c,
                    cfg,
                    cal_errs_raw[c],
                    rm_state_thresh=rm_state_thresh[c],
                    aci_state_alpha=aci_state_alpha[c],
                    volt_win=volt_wins[c],
                )
                rm_state_thresh[c] = next_rm
                aci_state_alpha[c] = next_aci
                step_det_maps.append(det_map)

                if cfg.repair_method_source in det_map:
                    _, src_sc, _ = det_map[cfg.repair_method_source]
                    win_anom_mask = src_sc > 0.0
                    sev_scores = src_sc
                else:
                    win_anom_mask = np.zeros(cfg.window_size, dtype=bool)
                    sev_scores = np.zeros(cfg.window_size, dtype=np.float64)
                    for m_name, (_, sc_arr, _) in det_map.items():
                        if m_name != CONSENSUS_METHOD_NAME:
                            win_anom_mask |= sc_arr > 0.0
                            sev_scores += (sc_arr > 0.0).astype(np.float64)

                rep_w, n_rep = repair_window_hybrid(
                    wins[c],
                    win_anom_mask,
                    cfg,
                    lstm_pred_series=pre_pred_c,
                    severity_scores=sev_scores,
                )
                step_rep_wins.append(rep_w)
                step_rep_counts.append(n_rep)

                agnostic_clean_mask = _build_model_agnostic_clean_mask(
                    wins[c], cfg.lstm_seq_len, det_map, cfg.clean_eval_guard_radius
                )
                step_trans_mask = _build_step_transition_mask(
                    wins[c], cfg.lstm_seq_len, cfg.transition_mad_mult, cfg.transition_guard_radius
                )
                pers_st = _compute_persistence_metrics(
                    wins[c], cfg.lstm_seq_len, agnostic_clean_mask, cfg.min_r2_variance
                )
                pers_stats_list.append(pers_st)

                _, _, st_raw_c = lstm_window_predict_and_metrics(
                    wins[c],
                    cur_raw_sd,
                    cfg,
                    eval_target=wins[c],
                    clean_eval_mask=agnostic_clean_mask,
                    transition_mask=step_trans_mask,
                    reusable_model=shared_model,
                )
                _, _, st_rep_c = lstm_window_predict_and_metrics(
                    rep_w,
                    cur_rep_sd,
                    cfg,
                    eval_target=wins[c],
                    clean_eval_mask=agnostic_clean_mask,
                    transition_mask=step_trans_mask,
                    reusable_model=shared_model,
                )
                _, _, st_net3_c = lstm_window_predict_and_metrics(
                    rep_w,
                    server_fedavg_net3_sd,
                    cfg,
                    eval_target=wins[c],
                    clean_eval_mask=agnostic_clean_mask,
                    transition_mask=step_trans_mask,
                    reusable_model=shared_model,
                )
                _, _, st_n1_on_rep = lstm_window_predict_and_metrics(
                    rep_w,
                    cur_raw_sd,
                    cfg,
                    eval_target=wins[c],
                    clean_eval_mask=agnostic_clean_mask,
                    reusable_model=shared_model,
                )
                _, _, st_n2_on_raw = lstm_window_predict_and_metrics(
                    wins[c],
                    cur_rep_sd,
                    cfg,
                    eval_target=wins[c],
                    clean_eval_mask=agnostic_clean_mask,
                    reusable_model=shared_model,
                )

                raw_net_stats.append(st_raw_c)
                rep_net_stats.append(st_rep_c)
                net3_srv_stats.append(st_net3_c)
                cross_ablation_list.append(
                    {
                        "net1_on_rep_mse": st_n1_on_rep["mse"],
                        "net2_on_raw_mse": st_n2_on_raw["mse"],
                    }
                )

                if cal_errs_raw[c] is None:
                    cal_errs_raw[c] = pre_err_c[agnostic_clean_mask] if np.any(agnostic_clean_mask) else pre_err_c
                else:
                    new_cal = pre_err_c[agnostic_clean_mask] if np.any(agnostic_clean_mask) else pre_err_c
                    cal_errs_raw[c] = np.concatenate([cal_errs_raw[c], new_cal])[-cfg.window_size * 2 :]

            epochs_cur = cfg.warmup_epochs if fl_step_idx == 1 else cfg.oos_epochs
            cached_raw_tensors = _prepare_cached_windows(wins, cfg)
            cached_rep_tensors = _prepare_cached_windows(step_rep_wins, cfg)

            raw_anom_fl_sd, raw_losses, raw_ep_done, raw_exchanged = federated_lstm_step(
                global_sd=raw_anom_fl_sd,
                client_windows=wins,
                mode=mode,
                cfg=cfg,
                epochs_to_run=epochs_cur,
                seed_offset=0,
                reusable_model=shared_model,
                local_models_sd=raw_local_sds if not exchange_flag else None,
                prepared_tensors=cached_raw_tensors,
            )

            repaired_fl_sd, rep_losses, rep_ep_done, _ = federated_lstm_step(
                global_sd=repaired_fl_sd,
                client_windows=step_rep_wins,
                mode=mode,
                cfg=cfg,
                epochs_to_run=epochs_cur,
                seed_offset=5000,
                reusable_model=shared_model,
                local_models_sd=rep_local_sds if not exchange_flag else None,
                prepared_tensors=cached_rep_tensors,
            )

            server_fedavg_net3_sd, net3_losses, net3_ep_done = canonical_server_fedavg_step(
                server_global_sd=server_fedavg_net3_sd,
                client_windows=step_rep_wins,
                cfg=cfg,
                epochs_to_run=epochs_cur,
                seed_offset=9000,
                reusable_model=shared_model,
                prepared_tensors=cached_rep_tensors,
            )

            for c in range(n_clients):
                st_raw_c = raw_net_stats[c]
                st_rep_c = rep_net_stats[c]
                st_n3_c = net3_srv_stats[c]
                st_p_c = pers_stats_list[c]
                st_x_c = cross_ablation_list[c]
                delta_mse = float(st_rep_c["mse"] - st_raw_c["mse"])
                lstm_metrics_records.append(
                    {
                        "seed": int(cfg.seed),
                        "fl_mode": mode,
                        "fl_exchange": int(exchange_flag),
                        "fl_step": fl_step_idx,
                        "is_prequential_oos": is_prequential_oos,
                        "client": c + 1,
                        "win_start": win_starts[c],
                        "win_end": win_starts[c] + cfg.window_size,
                        "repaired_points": step_rep_counts[c],
                        "scale_used": st_rep_c["scale_used"],
                        "epochs_net1": raw_ep_done[c],
                        "epochs_net2": rep_ep_done[c],
                        "epochs_net3_fedavg": net3_ep_done[c],
                        "target_var": st_raw_c["target_var"],
                        "pers_clean_mse": st_p_c["pers_clean_mse"],
                        "pers_clean_mae": st_p_c["pers_clean_mae"],
                        "pers_clean_r2": st_p_c["pers_clean_r2"],
                        "pers_all_valid_mse": st_p_c["pers_all_valid_mse"],
                        "raw_net_pts": st_raw_c["n_forecast_points"],
                        "raw_net_mse": st_raw_c["mse"],
                        "raw_net_rmse": st_raw_c["rmse"],
                        "raw_net_mae": st_raw_c["mae"],
                        "raw_net_r2": st_raw_c["r2"],
                        "raw_all_valid_mse": st_raw_c["all_valid_mse"],
                        "raw_all_valid_mae": st_raw_c["all_valid_mae"],
                        "transition_pts": st_raw_c["transition_pts"],
                        "raw_transition_mse": st_raw_c["transition_mse"],
                        "rep_net_pts": st_rep_c["n_forecast_points"],
                        "rep_net_mse": st_rep_c["mse"],
                        "rep_net_rmse": st_rep_c["rmse"],
                        "rep_net_mae": st_rep_c["mae"],
                        "rep_net_r2": st_rep_c["r2"],
                        "rep_all_valid_mse": st_rep_c["all_valid_mse"],
                        "rep_all_valid_mae": st_rep_c["all_valid_mae"],
                        "rep_transition_mse": st_rep_c["transition_mse"],
                        "fedavg_net3_pts": st_n3_c["n_forecast_points"],
                        "fedavg_net3_mse": st_n3_c["mse"],
                        "fedavg_net3_rmse": st_n3_c["rmse"],
                        "fedavg_net3_mae": st_n3_c["mae"],
                        "fedavg_net3_r2": st_n3_c["r2"],
                        "fedavg_net3_all_valid_mse": st_n3_c["all_valid_mse"],
                        "fedavg_net3_all_valid_mae": st_n3_c["all_valid_mae"],
                        "fedavg_net3_transition_mse": st_n3_c["transition_mse"],
                        "net1_on_rep_mse": st_x_c["net1_on_rep_mse"],
                        "net2_on_raw_mse": st_x_c["net2_on_raw_mse"],
                        "delta_mse_rep_minus_raw": delta_mse,
                    }
                )

            if write_logs:
                phase_tag = "OOS-eval" if is_prequential_oos else "Warm-up"
                ex_tag = "FL-sync: YES" if raw_exchanged else "FL-sync: NO (local)"
                comp_msg = " | ".join(
                    [
                        f"C{c + 1} (rep={step_rep_counts[c]}, pers={pers_stats_list[c]['pers_clean_mse']:.5f}, ep={raw_ep_done[c]}/{rep_ep_done[c]}/{net3_ep_done[c]}): "
                        f"Net1[raw]: MSE={raw_net_stats[c]['mse']:.5f}(all={raw_net_stats[c]['all_valid_mse']:.5f}, tr={raw_net_stats[c]['transition_mse']:.5f}), R2={raw_net_stats[c]['r2']:.4f} | "
                        f"Net2[{mode}]: MSE={rep_net_stats[c]['mse']:.5f}(all={rep_net_stats[c]['all_valid_mse']:.5f}, tr={rep_net_stats[c]['transition_mse']:.5f}), R2={rep_net_stats[c]['r2']:.4f} | "
                        f"Net3[Srv-FedAVG]: MSE={net3_srv_stats[c]['mse']:.5f}(all={net3_srv_stats[c]['all_valid_mse']:.5f}, tr={net3_srv_stats[c]['transition_mse']:.5f}), R2={net3_srv_stats[c]['r2']:.4f}"
                        for c in range(n_clients)
                    ]
                )
                print(f"  [{mode} | {ex_tag} | Step #{fl_step_idx} ({phase_tag})] -> {comp_msg}")
                if txt_log_file is not None:
                    txt_log_file.write(
                        f"  Step #{fl_step_idx} ({phase_tag}, {ex_tag}) | Window [{win_starts[0]}:{win_starts[0] + cfg.window_size}) | "
                        f"Loss1={[round(x, 6) for x in raw_losses]} | Loss2={[round(x, 6) for x in rep_losses]} | "
                        f"Loss3_SrvFedAvg={[round(x, 6) for x in net3_losses]} | {comp_msg}\n"
                    )

            for c in range(n_clients):
                w_lo = win_starts[c]
                w_hi = w_lo + cfg.window_size
                det_map = step_det_maps[c]
                st_raw_c = raw_net_stats[c]
                st_rep_c = rep_net_stats[c]
                st_n3_c = net3_srv_stats[c]

                rep_num[c][w_lo:w_hi] += step_rep_wins[c]
                rep_den[c][w_lo:w_hi] += 1.0

                for m_name, (lvl_arr, sc_arr, thr_val) in det_map.items():
                    anom_flag = (sc_arr > 0.0).astype(np.float64)
                    weighted_votes[c][m_name][w_lo:w_hi] += anom_flag
                    weighted_level_num[c][m_name][w_lo:w_hi] += lvl_arr
                    total_weights[c][m_name][w_lo:w_hi] += 1.0
                    thresh_acc[c][m_name] += float(thr_val)
                    thresh_cnt[c][m_name] += 1

                    if write_logs:
                        local_anom_idx = np.where(sc_arr > 0.0)[0]
                        global_anom_idx = (w_lo + local_anom_idx).tolist()
                        n_step_anom = len(global_anom_idx)
                        idx_preview = (
                            ",".join(map(str, global_anom_idx[:25]))
                            + (f",...(+{n_step_anom - 25})" if n_step_anom > 25 else "")
                        )
                        if txt_log_file is not None:
                            txt_log_file.write(
                                f"    Client {c + 1} | {m_name:<28} | detected: {n_step_anom:<4} | "
                                f"repaired: {step_rep_counts[c]:<4} | thr={thr_val:.4f} | indices=[{idx_preview}]\n"
                            )
                        if csv_writer is not None:
                            csv_writer.writerow(
                                [
                                    mode,
                                    int(exchange_flag),
                                    chunk_idx,
                                    fl_step_idx,
                                    int(is_prequential_oos),
                                    c + 1,
                                    w_lo,
                                    w_hi,
                                    step_rep_counts[c],
                                    raw_ep_done[c],
                                    rep_ep_done[c],
                                    net3_ep_done[c],
                                    round(float(raw_losses[c]), 6),
                                    round(float(rep_losses[c]), 6),
                                    round(float(net3_losses[c]), 6),
                                    st_raw_c["n_forecast_points"],
                                    round(float(st_raw_c["mse"]), 6),
                                    round(float(st_raw_c["rmse"]), 6),
                                    round(float(st_raw_c["mae"]), 6),
                                    round(float(st_raw_c["r2"]), 6),
                                    round(float(st_raw_c["all_valid_mse"]), 6),
                                    round(float(st_raw_c["transition_mse"]), 6) if np.isfinite(st_raw_c["transition_mse"]) else "nan",
                                    st_rep_c["n_forecast_points"],
                                    round(float(st_rep_c["mse"]), 6),
                                    round(float(st_rep_c["rmse"]), 6),
                                    round(float(st_rep_c["mae"]), 6),
                                    round(float(st_rep_c["r2"]), 6),
                                    round(float(st_rep_c["all_valid_mse"]), 6),
                                    round(float(st_rep_c["transition_mse"]), 6) if np.isfinite(st_rep_c["transition_mse"]) else "nan",
                                    st_n3_c["n_forecast_points"],
                                    round(float(st_n3_c["mse"]), 6),
                                    round(float(st_n3_c["rmse"]), 6),
                                    round(float(st_n3_c["mae"]), 6),
                                    round(float(st_n3_c["r2"]), 6),
                                    round(float(st_n3_c["all_valid_mse"]), 6),
                                    round(float(st_n3_c["transition_mse"]), 6) if np.isfinite(st_n3_c["transition_mse"]) else "nan",
                                    m_name,
                                    round(float(thr_val), 6),
                                    n_step_anom,
                                    ";".join(map(str, global_anom_idx)),
                                ]
                            )

                remaining_after_drop = len(buffers[c]) - cfg.step_size
                has_more_chunks = chunk_end < n_total
                if remaining_after_drop < cfg.window_size and not has_more_chunks:
                    buffers[c] = np.empty(0, dtype=np.float64)
                    volt_buffers[c] = np.empty(0, dtype=np.float64)
                else:
                    drop_n = min(cfg.step_size, len(buffers[c]))
                    buffers[c] = buffers[c][drop_n:]
                    if len(volt_buffers[c]) >= drop_n:
                        volt_buffers[c] = volt_buffers[c][drop_n:]
                    win_starts[c] += drop_n

            if write_logs and txt_log_file is not None:
                txt_log_file.flush()

            for c in range(n_clients):
                for m_name in active_methods:
                    weighted_votes[c][m_name] *= cfg.decay_gamma
                    weighted_level_num[c][m_name] *= cfg.decay_gamma
                    total_weights[c][m_name] *= cfg.decay_gamma

    final_masks: Dict[int, Dict[str, np.ndarray]] = {}
    method_levels: Dict[int, Dict[str, np.ndarray]] = {}
    method_thresh: Dict[int, Dict[str, float]] = {}
    stitched_repaired_signals: List[np.ndarray] = []

    for c in range(n_clients):
        final_masks[c] = {}
        method_levels[c] = {}
        method_thresh[c] = {}
        for m_name in active_methods:
            denom = total_weights[c][m_name]
            ratio = np.zeros(n_total, dtype=np.float64)
            lvl = np.zeros(n_total, dtype=np.float64)
            ok = denom > 1e-12
            ratio[ok] = weighted_votes[c][m_name][ok] / denom[ok]
            lvl[ok] = weighted_level_num[c][m_name][ok] / denom[ok]
            final_masks[c][m_name] = ratio >= cfg.vote_threshold
            method_levels[c][m_name] = lvl
            cnt = max(1, thresh_cnt[c][m_name])
            method_thresh[c][m_name] = thresh_acc[c][m_name] / cnt

        rep_sig_c = client_signals[c].copy()
        has_rep = rep_den[c] > 1e-12
        rep_sig_c[has_rep] = rep_num[c][has_rep] / rep_den[c][has_rep]
        stitched_repaired_signals.append(_sanitize_1d(rep_sig_c))

    final_raw_sds = raw_local_sds if not exchange_flag else [_clone_state_dict_cpu(raw_anom_fl_sd) for _ in range(n_clients)]
    final_rep_sds = rep_local_sds if not exchange_flag else [_clone_state_dict_cpu(repaired_fl_sd) for _ in range(n_clients)]

    return (
        final_masks,
        method_levels,
        method_thresh,
        stitched_repaired_signals,
        lstm_metrics_records,
        checkpoints_per_client,
        final_raw_sds,
        final_rep_sds,
    )


# ==============================================================================
# 8. VISUALIZATION (ENGLISH LABELS, CLEAN NETWORK CURVES WITHOUT ZERO-BASELINE)
# ==============================================================================
def plot_per_method_levels(
    signal: np.ndarray,
    masks: Dict[str, np.ndarray],
    levels: Dict[str, np.ndarray],
    thresholds: Dict[str, float],
    active_methods: List[str],
    client_idx: int,
    fl_mode: str,
    save_dir: Path,
    cfg: Config,
) -> None:
    if not cfg.save_per_method_plots and not cfg.show_per_method_plots:
        return
    x = np.arange(len(signal))
    for m_name in active_methods:
        mask = masks[m_name]
        lvl = levels[m_name]
        thr = thresholds[m_name]
        anom_idx = np.where(mask)[0]
        anom_sig_vals = signal[anom_idx]
        anom_lvl_vals = lvl[anom_idx]

        fig, (ax_sig, ax_lvl) = plt.subplots(2, 1, figsize=(14, 5.4), sharex=True)
        fig.suptitle(
            f"Client {client_idx + 1} | Method: {m_name} | Mode: {fl_mode}",
            fontsize=12,
            fontweight="bold",
        )

        ax_sig.plot(x, signal, color="tab:blue", linewidth=1.1, alpha=0.85, label="Observed Signal")
        if len(anom_idx) > 0:
            ax_sig.scatter(
                anom_idx,
                anom_sig_vals,
                color="crimson",
                s=24,
                zorder=5,
                label=f"Detected Anomalies ({len(anom_idx)} pts)",
            )
        ax_sig.set_ylabel("Signal Value", fontsize=9, fontweight="bold")
        ax_sig.grid(True, linestyle="--", alpha=0.35)
        ax_sig.legend(loc="upper right", fontsize=8)

        ax_lvl.plot(
            x,
            lvl,
            color="darkorange",
            linewidth=1.0,
            alpha=0.85,
            marker="o",
            markersize=2.5,
            label=f"Anomaly Score (n={len(x)})",
        )
        ax_lvl.axhline(
            thr,
            color="black",
            linestyle="--",
            linewidth=1.3,
            label=f"Threshold ({thr:.3f})",
        )
        if len(anom_idx) > 0:
            ax_lvl.scatter(
                anom_idx,
                anom_lvl_vals,
                color="crimson",
                s=26,
                zorder=6,
                label=f"Flagged Points ({len(anom_idx)} pts)",
            )
        ax_lvl.set_ylabel("Anomaly Score", fontsize=9, fontweight="bold")
        ax_lvl.set_xlabel("Sample Index", fontsize=10)
        ax_lvl.grid(True, linestyle="--", alpha=0.35)
        ax_lvl.legend(loc="upper right", fontsize=8)

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        if cfg.save_per_method_plots:
            safe_name = re.sub(r"[^\w]+", "_", m_name.lower()).strip("_")
            _safe_savefig(
                fig,
                save_dir / f"client_{client_idx + 1}_{fl_mode}_{safe_name}.png",
                cfg.fallback_results_dir,
                dpi=150,
            )
        if cfg.show_per_method_plots:
            plt.show()
        else:
            plt.close(fig)


def plot_two_fl_networks_comparison(
    lstm_recs: List[Dict[str, object]],
    fl_mode: str,
    save_dir: Path,
    cfg: Config,
) -> None:
    if not lstm_recs:
        return
    df = pd.DataFrame(lstm_recs)
    if "is_prequential_oos" in df.columns:
        df = df[df["is_prequential_oos"] == True]
    if df.empty:
        return

    clients = sorted(df["client"].unique())
    n_c = len(clients)
    fig, axes = plt.subplots(n_c, 1, figsize=(14, 3.6 * n_c), sharex=True)
    if n_c == 1:
        axes = [axes]

    for ax, c_id in zip(axes, clients):
        sub = df[df["client"] == c_id].sort_values("fl_step")
        if getattr(cfg, "plot_persistence_on_mse_curves", False) and "pers_clean_mse" in sub.columns:
            ax.plot(
                sub["fl_step"],
                sub["pers_clean_mse"],
                marker=".",
                linewidth=1.1,
                linestyle=":",
                color="gray",
                label="Persistence Baseline (Clean OOS MSE)",
            )
        ax.plot(
            sub["fl_step"],
            sub["raw_net_mse"],
            marker="o",
            linewidth=1.3,
            color="crimson",
            label="Net 1 (Raw Signal -> Clean OOS MSE)",
        )
        ax.plot(
            sub["fl_step"],
            sub["rep_net_mse"],
            marker="s",
            linewidth=1.3,
            color="seagreen",
            label=f"Net 2 (Repaired Signal, {fl_mode} -> Clean OOS MSE)",
        )
        if "fedavg_net3_mse" in sub.columns:
            ax.plot(
                sub["fl_step"],
                sub["fedavg_net3_mse"],
                marker="^",
                linewidth=1.3,
                linestyle="--",
                color="darkorange",
                label="Net 3 (Server FedAvg on Repaired -> Clean OOS MSE)",
            )
        ax.set_title(f"Client {c_id} | Mode: {fl_mode} — Prequential Out-of-Sample MSE Across Three Networks")
        ax.set_ylabel("Clean OOS MSE")
        ax.grid(True, linestyle="--", alpha=0.35)
        ax.legend(loc="upper right", fontsize=8)

    axes[-1].set_xlabel("Sliding Window Step (fl_step, out-of-sample)")
    plt.tight_layout()
    _safe_savefig(fig, save_dir / f"two_fl_networks_mse_{fl_mode}.png", cfg.fallback_results_dir, dpi=150)
    plt.show()


def plot_stitched_series_all_anomalies(
    client_signals: List[np.ndarray],
    repaired_signals: List[np.ndarray],
    masks_by_client: Dict[int, Dict[str, np.ndarray]],
    active_methods: List[str],
    fl_mode: str,
    cfg: Config,
    save_dir: Path,
) -> Dict[int, Dict[str, int]]:
    n_clients = len(client_signals)
    n_total = len(client_signals[0])
    x = np.arange(n_total)

    fig, axes = plt.subplots(n_clients, 1, figsize=(15, 4.4 * n_clients), sharex=True)
    if n_clients == 1:
        axes = [axes]

    cmap = plt.get_cmap("tab20")
    counts_summary: Dict[int, Dict[str, int]] = {}
    core_present = [m for m in CORE_ROBUST_METHODS if m in active_methods]

    for c_idx, (ax, sig, rep_sig) in enumerate(zip(axes, client_signals, repaired_signals)):
        counts_summary[c_idx] = {}
        ax.plot(x, sig, color="steelblue", linewidth=1.0, alpha=0.75, label="Stitched Signal (Raw)")
        ax.plot(
            x,
            rep_sig,
            color="seagreen",
            linewidth=1.0,
            linestyle="--",
            alpha=0.80,
            label="Stitched Signal (Repaired)",
        )

        for b_pos in range(cfg.step_size, n_total, cfg.step_size):
            ax.axvline(b_pos, color="gray", linestyle=":", linewidth=0.8, alpha=0.5)

        distinct_core_mask, _ = compute_distinct_anomalies_mask(masks_by_client[c_idx], core_present, n_total)
        distinct_all_mask, _ = compute_distinct_anomalies_mask(masks_by_client[c_idx], active_methods, n_total)
        consensus_count = int(np.sum(masks_by_client[c_idx].get(CONSENSUS_METHOD_NAME, np.zeros(n_total, dtype=bool))))

        for m_idx, m_name in enumerate(active_methods):
            m_mask = masks_by_client[c_idx][m_name]
            idx_m = np.where(m_mask)[0]
            counts_summary[c_idx][m_name] = int(len(idx_m))
            if len(idx_m) > 0:
                ax.scatter(
                    idx_m,
                    sig[idx_m],
                    color=cmap(m_idx % 20),
                    s=22,
                    alpha=0.88,
                    zorder=4 + m_idx,
                    label=f"{m_name}: {len(idx_m)}",
                )

        total_core_distinct = int(np.sum(distinct_core_mask))
        total_all_distinct = int(np.sum(distinct_all_mask))
        counts_summary[c_idx]["consensus_anomalies"] = consensus_count
        counts_summary[c_idx]["distinct_anomalies_core"] = total_core_distinct
        counts_summary[c_idx]["distinct_anomalies_all"] = total_all_distinct

        ax.set_title(
            f"Client {c_idx + 1} | Mode: {fl_mode} — Stitched Series (n={n_total}) | "
            f"Consensus: {consensus_count} | Distinct (Core): {total_core_distinct} | Distinct (All): {total_all_distinct}",
            fontsize=11,
            fontweight="bold",
        )
        ax.set_ylabel("Signal Value", fontsize=9, fontweight="bold")
        ax.grid(True, linestyle="--", alpha=0.35)
        ax.legend(loc="upper right", fontsize=8, ncol=3)

    axes[-1].set_xlabel("Sample Index Across Stitched Series", fontsize=10)
    plt.tight_layout()
    _safe_savefig(fig, save_dir / f"stitched_all_anomalies_{fl_mode}.png", cfg.fallback_results_dir, dpi=150)
    plt.show()

    return counts_summary


def plot_methods_effectiveness_benchmark(
    eff_df: pd.DataFrame,
    save_dir: Path,
    fallback_dir: Path = Path("/tmp/AWMAD_Results"),
) -> None:
    if eff_df.empty:
        return
    sub = eff_df[eff_df["fl_mode"] == "fedprox_inv_loss"]
    if sub.empty:
        sub = eff_df.iloc[: min(len(eff_df), 13)]
    sub = sub.sort_values("mean_f1", ascending=True)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.6), sharey=True)
    y_pos = np.arange(len(sub))
    colors = [
        "seagreen" if m == CONSENSUS_METHOD_NAME else ("steelblue" if flag else "slategray")
        for m, flag in zip(sub["method"], sub["is_core_robust"])
    ]

    ax1.barh(y_pos, sub["mean_f1"], color=colors, alpha=0.88)
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels(sub["method"], fontsize=9)
    ax1.set_xlabel("Mean Point-Level F1 on Control Anomalies")
    ax1.set_title("Detector Effectiveness by Point F1")
    ax1.grid(True, axis="x", linestyle="--", alpha=0.35)

    ax2.barh(y_pos, sub["mean_neg_fpr"], color=colors, alpha=0.72)
    ax2.set_xlabel("Mean False Positive Rate on Negative Windows (Lower = Better)")
    ax2.set_title("Negative Control False Positive Rate (Neg-FPR)")
    ax2.grid(True, axis="x", linestyle="--", alpha=0.35)

    plt.tight_layout()
    _safe_savefig(fig, save_dir / "benchmark_all_methods_f1_fpr.png", fallback_dir, dpi=150)
    plt.show()


def plot_fl_modes_comparison_summary(
    fl_sum_df: pd.DataFrame,
    save_dir: Path,
    fallback_dir: Path = Path("/tmp/AWMAD_Results"),
) -> None:
    if fl_sum_df.empty:
        return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.2))
    modes = ["local_only", "fedavg", "fedprox_inv_loss"]
    present_modes = [m for m in modes if m in fl_sum_df["fl_mode"].unique()]
    if not present_modes:
        return

    x = np.arange(len(present_modes))
    width = 0.25

    for idx_c, c_id in enumerate([1, 2, 3]):
        vals = []
        for m in present_modes:
            row = fl_sum_df[(fl_sum_df["fl_mode"] == m) & (fl_sum_df["client"] == c_id)]
            vals.append(float(row["net2_vs_net1_impr_pct"].iloc[0]) if not row.empty else 0.0)
        ax1.bar(x + (idx_c - 1) * width, vals, width=width, label=f"Client {c_id} (Net 2)")

    ax1.axhline(0.0, color="black", linewidth=0.9)
    ax1.set_xticks(x)
    ax1.set_xticklabels(present_modes, fontsize=10)
    ax1.set_ylabel("Clean OOS MSE Reduction vs. Net 1 (%)")
    ax1.set_title("Net 2 Clean OOS MSE Reduction vs. Net 1 (3 Seeds)")
    ax1.grid(True, axis="y", linestyle="--", alpha=0.35)
    ax1.legend(fontsize=8)

    for idx_c, c_id in enumerate([1, 2, 3]):
        n3_vals = []
        for m in present_modes:
            row = fl_sum_df[(fl_sum_df["fl_mode"] == m) & (fl_sum_df["client"] == c_id)]
            n3_vals.append(float(row["net3_fedavg_vs_net1_impr_pct"].iloc[0]) if not row.empty else 0.0)
        ax2.bar(x + (idx_c - 1) * width, n3_vals, width=width, color=["#d95f02", "#e6ab02", "#a6761d"][idx_c], label=f"Client {c_id} (Net 3)")

    ax2.axhline(0.0, color="black", linewidth=0.9)
    ax2.set_xticks(x)
    ax2.set_xticklabels(present_modes, fontsize=10)
    ax2.set_ylabel("Clean OOS MSE Reduction vs. Net 1 (%)")
    ax2.set_title("Net 3 (Server FedAvg on Repaired) vs. Net 1 (3 Seeds)")
    ax2.grid(True, axis="y", linestyle="--", alpha=0.35)
    ax2.legend(fontsize=8)

    plt.tight_layout()
    _safe_savefig(fig, save_dir / "fl_modes_net1_vs_net2_summary.png", fallback_dir, dpi=150)
    plt.show()


def plot_repair_zoom_example(
    signal: np.ndarray,
    repaired_signal: np.ndarray,
    consensus_mask: np.ndarray,
    fl_mode: str,
    save_dir: Path,
    fallback_dir: Path = Path("/tmp/AWMAD_Results"),
) -> None:
    n = len(signal)
    if n < 60 or not np.any(consensus_mask):
        return

    win_zoom = min(240, n)
    counts_roll = pd.Series(consensus_mask.astype(int)).rolling(window=win_zoom, min_periods=1).sum().to_numpy()
    best_end = int(np.argmax(counts_roll))
    lo = max(0, best_end - win_zoom + 1)
    hi = min(n, lo + win_zoom)
    xs = np.arange(lo, hi)

    fig, ax = plt.subplots(figsize=(14, 4.2))
    ax.plot(xs, signal[lo:hi], color="steelblue", linewidth=1.2, label="Raw Signal with Anomalies")
    ax.plot(
        xs,
        repaired_signal[lo:hi],
        color="seagreen",
        linewidth=1.4,
        linestyle="--",
        label="Repaired Signal (Net 2 & Net 3 Input)",
    )
    anom_sub = np.where(consensus_mask[lo:hi])[0] + lo
    if len(anom_sub) > 0:
        ax.scatter(
            anom_sub,
            signal[anom_sub],
            color="crimson",
            s=28,
            zorder=5,
            label=f"Consensus Anomalies in Zoom ({len(anom_sub)})",
        )
    ax.set_title(f"Client 1 | Mode: {fl_mode} — Detailed Repair Zoom [{lo}:{hi})")
    ax.set_xlabel("Sample Index")
    ax.set_ylabel("Target Signal Value")
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    _safe_savefig(fig, save_dir / f"repair_zoom_and_consensus_effect_{fl_mode}.png", fallback_dir, dpi=150)
    plt.show()


def write_q1_manuscript_summary(
    q1_path: Path,
    base_cfg: Config,
    active_methods: List[str],
    explicitly_excluded: List[str],
    fl_comp_df: pd.DataFrame,
    w_ms_df: pd.DataFrame,
    scen_df: pd.DataFrame,
    qss_df: pd.DataFrame,
) -> None:
    with open(q1_path, "w", encoding="utf-8") as f:
        f.write("=== IEEE / MDPI Q1 MANUSCRIPT SUMMARY & LIMITATIONS ===\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Modes: {base_cfg.fl_modes} | Seeds: {base_cfg.seeds}\n")
        f.write(f"Active methods ({len(active_methods)}): {active_methods}\n")
        f.write(f"Explicitly excluded methods ({len(explicitly_excluded)}): {explicitly_excluded}\n\n")

        if not fl_comp_df.empty:
            f.write("--- CROSS-MODE COMPARISON (3 SEEDS, CLEAN / ALL-VALID / STEP-TRANSITION) ---\n")
            f.write(fl_comp_df.to_string(index=False) + "\n\n")

        if not w_ms_df.empty:
            f.write("--- MULTI-SEED WILCOXON TEST WITH HOLM CORRECTION ---\n")
            f.write(w_ms_df.to_string(index=False) + "\n\n")

        if not scen_df.empty:
            f.write("--- CONTROL VALIDATION BY SCENARIO (SPIKES VS ZERO-OUT SEGMENT) ---\n")
            f.write(scen_df.to_string(index=False) + "\n\n")

        if not qss_df.empty:
            f.write("--- QSS METHOD SELECTION TABLE ---\n")
            f.write(qss_df.to_string(index=False) + "\n\n")

        f.write("--- KEY METHODOLOGICAL LIMITATIONS FOR Q1 DISCUSSION ---\n")
        f.write("1. Forecast model is univariate residual LSTM on target lags; voltage is used only in consensus regime gating.\n")
        f.write("2. Windowed repair uses neighboring window context and is not strictly causal in real-time streaming.\n")
        f.write(f"3. Repair ratio is capped at {base_cfg.max_repair_ratio:.0%} per window, limiting correction on prolonged shifts or drifts.\n")
        f.write("4. Clean OOS MSE, all-valid MSE, step-transition MSE, and persistence baseline are reported jointly in tables.\n")
        f.write("5. Natural anomalies lack manual event-level ground truth; control metrics rely on injected spikes and zero-out segments.\n")


# ==============================================================================
# 9. MAIN ENTRY POINT
# ==============================================================================
def main() -> None:
    base_cfg = Config()
    _set_global_seeds(base_cfg.seed)
    t_start = time.time()

    active_methods = get_active_methods(base_cfg)
    explicitly_excluded = get_explicitly_excluded_methods(base_cfg)
    if not active_methods:
        raise ValueError("No methods enabled in Config.enabled_methods.")

    base_dir, results_dir, _, signals_1d, voltages_1d, voltage_col, target_col, usable_len = load_and_prepare_clients(base_cfg)
    print(f"Input directory: {base_dir}")
    print(f"Results directory: {results_dir}")
    print(f"Active methods ({len(active_methods)}): {active_methods}")
    print(f"Explicitly excluded noisy/inactive methods ({len(explicitly_excluded)}): {explicitly_excluded}")
    print(f"Voltage (gate-only): {voltage_col} | Target (LSTM): {target_col} | Usable length: {usable_len}")
    print(f"Modes: {base_cfg.fl_modes} | Seeds: {base_cfg.seeds}")

    if usable_len < base_cfg.window_size:
        print(f"Insufficient points ({usable_len}) for window_size={base_cfg.window_size}.")
        return

    txt_log_path = results_dir / base_cfg.log_txt_name
    csv_log_path = results_dir / base_cfg.log_csv_name
    ctrl_csv_path = results_dir / base_cfg.ctrl_csv_name
    ctrl_summary_csv_path = results_dir / base_cfg.ctrl_summary_csv_name
    wilcoxon_csv_path = results_dir / base_cfg.wilcoxon_csv_name
    methods_eff_csv_path = results_dir / base_cfg.methods_eff_csv_name
    fl_summary_csv_path = results_dir / base_cfg.fl_summary_csv_name
    excluded_csv_path = results_dir / base_cfg.excluded_methods_csv_name
    q1_summary_path = results_dir / base_cfg.q1_summary_txt_name

    seeds_list = list(base_cfg.seeds) if base_cfg.seeds else [base_cfg.seed]
    primary_seed = seeds_list[0]

    with open(txt_log_path, "w", encoding="utf-8") as txt_f, open(
        csv_log_path, "w", newline="", encoding="utf-8"
    ) as csv_f:
        txt_f.write(
            f"=== DETECTION, REPAIR & THREE-NETWORK LSTM LOG (PREQUENTIAL OOS + MULTI-SEED, "
            f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}) ===\n"
            f"Data dir: {base_dir}\n"
            f"Results dir: {results_dir}\n"
            f"Modes: {base_cfg.fl_modes} | Seeds: {seeds_list}\n"
            f"IF n_estimators={base_cfg.if_estimators} | Scale guard: MAD+IQR+{base_cfg.robust_scale_std_ratio}*std (floor={base_cfg.robust_scale_min_floor})\n"
            f"Active methods ({len(active_methods)}): {active_methods}\n"
            f"Explicitly excluded noisy/inactive methods ({len(explicitly_excluded)}): {explicitly_excluded}\n"
            f"FL params: fedprox_mu={base_cfg.fedprox_mu}, tau={base_cfg.fedprox_loss_tau}, "
            f"fl_max_update_norm={base_cfg.fl_max_update_norm}, local_max_update_norm={base_cfg.local_max_update_norm}\n"
        )

        csv_writer = csv.writer(csv_f)
        csv_writer.writerow(
            [
                "fl_mode",
                "fl_exchange",
                "chunk_idx",
                "fl_step_idx",
                "is_prequential_oos",
                "client_id",
                "win_start",
                "win_end",
                "repaired_points",
                "epochs_net1",
                "epochs_net2",
                "epochs_net3_fedavg",
                "train_loss_net1_anom",
                "train_loss_net2_repaired",
                "train_loss_net3_srv_fedavg",
                "net1_anom_forecast_points",
                "net1_anom_clean_mse",
                "net1_anom_clean_rmse",
                "net1_anom_clean_mae",
                "net1_anom_clean_r2",
                "net1_all_valid_mse",
                "net1_transition_mse",
                "net2_rep_forecast_points",
                "net2_rep_clean_mse",
                "net2_rep_clean_rmse",
                "net2_rep_clean_mae",
                "net2_rep_clean_r2",
                "net2_all_valid_mse",
                "net2_transition_mse",
                "net3_fedavg_forecast_points",
                "net3_fedavg_clean_mse",
                "net3_fedavg_clean_rmse",
                "net3_fedavg_clean_mae",
                "net3_fedavg_clean_r2",
                "net3_all_valid_mse",
                "net3_transition_mse",
                "method",
                "threshold",
                "step_anomalies_count",
                "global_anomaly_indices",
            ]
        )

        all_ctrl_frames: List[pd.DataFrame] = []
        all_wilcoxon_frames: List[pd.DataFrame] = []
        all_oos_frames: List[pd.DataFrame] = []
        all_hi_frames: List[pd.DataFrame] = []
        stream_counts_rows: List[Dict[str, object]] = []

        for mode in base_cfg.fl_modes:
            ex_flag = mode_exchanges_params(mode)
            ex_label = "YES (weight aggregation across clients)" if ex_flag else "NO (isolated local training)"
            print(f"\n--- Mode: {mode} | FL weight exchange: {ex_label} ---")
            txt_f.write("\n====================================================================\n")
            txt_f.write(f"TRAINING MODE: {mode} | FL EXCHANGE: {ex_label}\n")
            txt_f.write("====================================================================\n")

            mode_lstm_recs_all_seeds: List[Dict[str, object]] = []
            primary_outputs = None

            for s_idx, s_val in enumerate(seeds_list):
                cfg_s = Config(**{f.name: getattr(base_cfg, f.name) for f in base_cfg.__dataclass_fields__.values()})
                cfg_s.seed = int(s_val)
                is_primary = s_idx == 0
                if cfg_s.save_per_method_plots_only_primary_seed and not is_primary:
                    cfg_s.save_per_method_plots = False

                out_s = run_streaming_fl_and_detection(
                    signals_1d,
                    voltage_signals=voltages_1d,
                    mode=mode,
                    cfg=cfg_s,
                    txt_log_file=txt_f,
                    csv_writer=csv_writer,
                    write_logs=is_primary,
                )
                if is_primary:
                    primary_outputs = out_s

                mode_lstm_recs_all_seeds.extend(out_s[4])

                w_df_s = compute_wilcoxon_summary(out_s[4], mode=mode)
                if not w_df_s.empty:
                    all_wilcoxon_frames.append(w_df_s)

                if cfg_s.run_control_validation:
                    ctrl_df_s = run_controlled_anomaly_validation(
                        raw_signals=signals_1d,
                        voltage_signals=voltages_1d,
                        checkpoints_by_client_and_start=out_s[5],
                        fallback_raw_sds=out_s[6],
                        fallback_rep_sds=out_s[7],
                        cfg=cfg_s,
                    )
                    if not ctrl_df_s.empty:
                        ctrl_df_s.insert(0, "fl_mode", mode)
                        all_ctrl_frames.append(ctrl_df_s)

            assert primary_outputs is not None
            (
                masks_by_client,
                levels_by_client,
                thresh_by_client,
                repaired_signals_1d,
                _,
                _,
                _,
                _,
            ) = primary_outputs

            if mode_lstm_recs_all_seeds:
                lstm_df = pd.DataFrame(mode_lstm_recs_all_seeds)
                oos_df = lstm_df[lstm_df["is_prequential_oos"] == True] if "is_prequential_oos" in lstm_df.columns else lstm_df
                if not oos_df.empty:
                    all_oos_frames.append(oos_df)

                lstm_summary = (
                    oos_df.groupby(["fl_mode", "fl_exchange", "client"], as_index=False)
                    .agg(
                        n_seeds=("seed", "nunique"),
                        oos_records=("fl_step", "count"),
                        total_repaired_pts=("repaired_points", "sum"),
                        pers_clean_mse=("pers_clean_mse", "mean"),
                        net1_oos_mse=("raw_net_mse", "mean"),
                        net2_oos_mse=("rep_net_mse", "mean"),
                        net3_fedavg_oos_mse=("fedavg_net3_mse", "mean"),
                        net1_all_valid_mse=("raw_all_valid_mse", "mean"),
                        net2_all_valid_mse=("rep_all_valid_mse", "mean"),
                        net3_all_valid_mse=("fedavg_net3_all_valid_mse", "mean"),
                        net1_trans_mse=("raw_transition_mse", "mean"),
                        net2_trans_mse=("rep_transition_mse", "mean"),
                        net3_trans_mse=("fedavg_net3_transition_mse", "mean"),
                        net1_oos_r2=("raw_net_r2", "mean"),
                        net2_oos_r2=("rep_net_r2", "mean"),
                        net3_fedavg_oos_r2=("fedavg_net3_r2", "mean"),
                    )
                )
                print(
                    f"\nPrequential OOS summary ({len(seeds_list)} seeds) across clean, all-valid & transitions ({mode}):"
                )
                print(lstm_summary.to_string(index=False))
                txt_f.write(f"\n--- PREQUENTIAL OOS SUMMARY ({mode}) ---\n")
                txt_f.write(lstm_summary.to_string(index=False) + "\n")

                hi_df = summarize_high_anomaly_oos(mode_lstm_recs_all_seeds, mode=mode)
                if not hi_df.empty:
                    all_hi_frames.append(hi_df)
                    print(f"\nStratified OOS analysis on higher-repaired-count windows ({mode}):")
                    print(hi_df.to_string(index=False))
                    txt_f.write(f"\n--- STRATIFIED OOS ANALYSIS ON HIGHER-REPAIRED-COUNT WINDOWS ({mode}) ---\n")
                    txt_f.write(hi_df.to_string(index=False) + "\n")

                plot_two_fl_networks_comparison(
                    lstm_recs=[r for r in mode_lstm_recs_all_seeds if r["seed"] == primary_seed],
                    fl_mode=mode,
                    save_dir=results_dir,
                    cfg=base_cfg,
                )

            for c_idx, sig in enumerate(signals_1d):
                plot_per_method_levels(
                    signal=sig,
                    masks=masks_by_client[c_idx],
                    levels=levels_by_client[c_idx],
                    thresholds=thresh_by_client[c_idx],
                    active_methods=active_methods,
                    client_idx=c_idx,
                    fl_mode=mode,
                    save_dir=results_dir,
                    cfg=base_cfg,
                )

            plot_repair_zoom_example(
                signal=signals_1d[0],
                repaired_signal=repaired_signals_1d[0],
                consensus_mask=masks_by_client[0].get(CONSENSUS_METHOD_NAME, np.zeros(usable_len, dtype=bool)),
                fl_mode=mode,
                save_dir=results_dir,
                fallback_dir=base_cfg.fallback_results_dir,
            )

            counts_summary = plot_stitched_series_all_anomalies(
                client_signals=signals_1d,
                repaired_signals=repaired_signals_1d,
                masks_by_client=masks_by_client,
                active_methods=active_methods,
                fl_mode=mode,
                cfg=base_cfg,
                save_dir=results_dir,
            )

            summary_rows = []
            for c_idx, m_dict in counts_summary.items():
                row = {"fl_mode": mode, "client": f"Client {c_idx + 1}", **m_dict}
                summary_rows.append(row)
                stream_counts_rows.append(row)
            summary_df = pd.DataFrame(summary_rows)

            print(f"\nDistinct anomaly counts ({mode}, primary seed={primary_seed}):")
            print(summary_df.to_string(index=False))

            txt_f.write(f"\n--- DISTINCT ANOMALY COUNTS ({mode}) ---\n")
            txt_f.write(summary_df.to_string(index=False) + "\n")
            txt_f.flush()

        print_real_networks_and_fl_focus(all_oos_frames)

        ctrl_sum_df = pd.DataFrame()
        scen_df = pd.DataFrame()
        if all_ctrl_frames:
            ctrl_cat = pd.concat(all_ctrl_frames, ignore_index=True)
            ctrl_cat.to_csv(ctrl_csv_path, index=False)
            ctrl_sum_df = summarize_control_methods(ctrl_cat)
            if not ctrl_sum_df.empty:
                ctrl_sum_df.to_csv(ctrl_summary_csv_path, index=False)
            scen_df = summarize_control_by_scenario(ctrl_cat)
            if not scen_df.empty:
                print("\nControl validation breakdown by scenario (spikes_only vs zero_out_segment):")
                print(scen_df.to_string(index=False))
                txt_f.write("\n--- CONTROL VALIDATION BY SCENARIO ---\n")
                txt_f.write(scen_df.to_string(index=False) + "\n")

        w_ms_df = pd.DataFrame()
        if all_wilcoxon_frames:
            w_cat = pd.concat(all_wilcoxon_frames, ignore_index=True)
            w_cat.to_csv(wilcoxon_csv_path, index=False)
            w_ms_df = summarize_multiseed_wilcoxon(w_cat)
            if not w_ms_df.empty:
                print("\nAggregated Wilcoxon test with Holm correction across all seeds:")
                print(w_ms_df.to_string(index=False))
                txt_f.write("\n--- AGGREGATED WILCOXON-HOLM TEST ACROSS ALL SEEDS ---\n")
                txt_f.write(w_ms_df.to_string(index=False) + "\n")

        qss_df = pd.DataFrame()
        eff_df = build_methods_effectiveness_matrix(ctrl_sum_df, stream_counts_rows, usable_len)
        if not eff_df.empty:
            eff_df.to_csv(methods_eff_csv_path, index=False)
            print(f"\nMethods effectiveness matrix saved to: {methods_eff_csv_path}")
            print(eff_df.to_string(index=False))
            plot_methods_effectiveness_benchmark(eff_df, results_dir, base_cfg.fallback_results_dir)

            kept_math, qss_df = filter_methods_mathematically(eff_df, base_cfg)
            if not qss_df.empty:
                print("\nMathematical QSS evaluation and detector selection:")
                print(qss_df.to_string(index=False))
                print(f"Recommended methods by QSS ({len(kept_math)}): {kept_math}")
                txt_f.write("\n--- QSS EVALUATION & METHOD SELECTION ---\n")
                txt_f.write(qss_df.to_string(index=False) + "\n")

        excl_df = build_excluded_methods_report(base_cfg, qss_df)
        if not excl_df.empty:
            excl_df.to_csv(excluded_csv_path, index=False)
            print(f"\nExcluded noisy/inactive methods report saved to: {excluded_csv_path}")
            print(excl_df.to_string(index=False))
            txt_f.write("\n--- EXCLUDED NOISY / INACTIVE METHODS & REASONS ---\n")
            txt_f.write(excl_df.to_string(index=False) + "\n")

        fl_comp_df = build_fl_modes_comparison_summary(all_oos_frames, all_hi_frames, w_ms_df)
        if not fl_comp_df.empty:
            fl_comp_df.to_csv(fl_summary_csv_path, index=False)
            print(f"\nCross-mode comparison table across three networks saved to: {fl_summary_csv_path}")
            print(fl_comp_df.to_string(index=False))
            txt_f.write("\n--- CROSS-MODE COMPARISON TABLE ACROSS THREE NETWORKS ---\n")
            txt_f.write(fl_comp_df.to_string(index=False) + "\n")
            plot_fl_modes_comparison_summary(fl_comp_df, results_dir, base_cfg.fallback_results_dir)

        write_q1_manuscript_summary(
            q1_summary_path,
            base_cfg,
            active_methods,
            explicitly_excluded,
            fl_comp_df,
            w_ms_df,
            scen_df,
            qss_df,
        )

    _sync_fallback_to_results(base_cfg.fallback_results_dir, results_dir)
    print(f"\nDone in {time.time() - t_start:.1f} s.")
    print(f"Text log saved to: {txt_log_path}")
    print(f"Step CSV log saved to: {csv_log_path}")
    print(f"Q1 manuscript summary saved to: {q1_summary_path}")


if __name__ == "__main__":
    main()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:

import os
import shutil
import zipfile
from pathlib import Path
from google.colab import drive

# Підключення Google Drive
drive.mount('/content/drive')

# Шляхи
SOURCE = Path("/content/drive/MyDrive/DATA/Results")
OUTPUT = Path("/content/Results_Gallery")
IMAGES = OUTPUT / "images"

# Створення папок
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)

IMAGES.mkdir(parents=True)

# Пошук та копіювання PNG
png_files = sorted(
    p for p in SOURCE.rglob("*")
    if p.is_file() and p.suffix.lower() == ".png"
)

for file in png_files:
    relative = file.relative_to(SOURCE)
    destination = IMAGES / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(file, destination)

# Формування HTML
html = """
<!DOCTYPE html>
<html lang="uk">
<head>
<meta charset="UTF-8">

<style>
body {
    margin: 10px;
    background: #eeeeee;
}

.gallery {
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(300px, 1fr));
    gap: 10px;
}

.gallery img {
    width: 100%;
    height: 280px;
    object-fit: contain;
    background: white;
    cursor: pointer;
}

</style>
</head>

<body>

<div class="gallery">
"""

for file in sorted(IMAGES.rglob("*")):
    if file.is_file() and file.suffix.lower() == ".png":
        relative = file.relative_to(OUTPUT).as_posix()
        html += f'<img src="{relative}" loading="lazy">\n'

html += """
</div>

</body>
</html>
"""

# Збереження HTML
with open(OUTPUT / "index.html", "w", encoding="utf-8") as f:
    f.write(html)

# Архів для завантаження
with zipfile.ZipFile(
    "/content/Results_Gallery.zip",
    "w",
    zipfile.ZIP_DEFLATED
) as archive:

    for file in OUTPUT.rglob("*"):
        if file.is_file():
            archive.write(file, file.relative_to(OUTPUT.parent))

print("PNG:", len(png_files))
print("Готово!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PNG: 83
Готово!


In [ ]:
from google.colab import files
files.download("/content/Results_Gallery.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import base64
import mimetypes
from IPython.display import display, HTML
from pathlib import Path

image_folder = Path("/content/Results_Gallery/images")

gallery = """
<div style="
    display:grid;
    grid-template-columns:repeat(auto-fill,minmax(300px,1fr));
    gap:10px;
">
"""

for file in sorted(image_folder.rglob("*.png")):
    with open(file, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    gallery += f"""
    <img
        src="data:image/png;base64,{encoded}"
        style="width:100%;height:280px;object-fit:contain;background:white;"
    >
    """

gallery += "</div>"

display(HTML(gallery))


Output hidden; open in https://colab.research.google.com to view.